# Compound Facial Emotion Recognition — Full Comparison Framework

Consolidated notebook: Morphological Splicing + SA-CNN + ALSTM + ViT (Proposed)
vs. 6 baseline architectures, trained/evaluated under identical conditions.

**Kaggle setup, before you hit Run All:**
1. Settings (right sidebar) → Accelerator → **GPU P100**.
2. Add Data → attach your RAF-DB / FER-2013 / JAFFE / CK+ / AffectNet datasets
   (whatever slugs you uploaded them under on Kaggle).
3. In the **Configuration** cell below, update `RAF_DB_DIR`, `FER2013_DIR`,
   `JAFFE_DIR`, `CKP_DIR`, `AFFECTNET_DIR` to match the exact paths Kaggle
   mounts your datasets at (check the "Input" panel on the right once
   they're attached — it shows the real `/kaggle/input/...` path).
4. Everything else — all 7 model architectures, the 3-stage training
   curriculum, dataset loading + morphological splicing, evaluation metrics,
   plots, benchmarking, comparison tables, ablation study, and dataset-wise
   evaluation (including FER-2013) — is filled in and ready to run top to
   bottom (Run All).

**Note on runtime:** this trains 7 models × 3 stages each on a single P100.
Kaggle sessions cap out at 9h (12h for some accounts) of *continuous*
GPU time. If you can't finish all 7 in one session, edit `MODEL_LIST` in the
"Run Everything" cell near the bottom to train a subset per session — every
stage saves checkpoints/logs to `/kaggle/working/`, so you can resume across
sessions as long as you download and re-upload `/kaggle/working/checkpoints`
and `/kaggle/working/logs` (or save them as a Kaggle Dataset) between runs.


## Setup

In [ ]:
# ── CELL 1a — Install packages (safe for both interactive AND Save & Run All) ──
# NOTE: the kernel-restart cell that used to follow this one is REMOVED.
# It broke "Save & Run All (Commit)" runs: a commit executes on a brand-new,
# never-imported kernel, so an explicit restart mid-run just kills the whole
# batch job -- Kaggle's committer sees the kernel disconnect and stops
# instead of continuing to the next cell. Since nothing has imported torch
# yet in a fresh commit session, Cell 1b's `import torch` below will pick up
# this pinned version naturally, with no restart needed.
#
# (If you're editing interactively and had an OLDER torch already loaded
# in this same live session from earlier experimentation, you may still
# want to manually restart the kernel via the toolbar before running Cell
# 1b -- but that's a manual interactive choice, not something this notebook
# should trigger automatically during a commit.)

import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.4.1+cu118",
    "torchvision==0.19.1+cu118",
    "torchaudio==2.4.1+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118"
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm", "albumentations", "seaborn", "opencv-python-headless"
])

print("✅ Installation complete.")
print(">>> Continuing directly to CELL 1b (imports) -- no restart needed for Save & Run All.")


In [ ]:
# ── CELL 1a.5 — (removed) ───────────────────────────────────────────────────
# The kernel restart previously here has been removed -- it broke Save & Run
# All (Commit) runs by disconnecting the kernel mid-batch. See the note in
# Cell 1a above. This cell is left as a no-op placeholder so cell numbering
# / any cross-references elsewhere in the notebook stay stable; it's safe to
# delete outright if you prefer.
print("ℹ️  Kernel restart skipped intentionally -- see note in Cell 1a. Proceeding to Cell 1b.")


In [ ]:
# ── CELL 1b — Imports (run after kernel restart) ──────────────────────────────
import random, time, json, shutil, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import (Dataset, DataLoader, ConcatDataset,
                               WeightedRandomSampler, random_split)
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch     : {torch.__version__}")
print(f"GPU         : {torch.cuda.get_device_name(0)}")
print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("✅ Imports complete")

## Configuration

In [ ]:
class Config:
    # ── Dataset paths — matched to your "Dataset2" folder tree ──────────────
    # Confirmed via os.walk: this account mounts datasets under
    # /kaggle/input/datasets/<username>/<dataset-slug>/, not the shorter
    # /kaggle/input/<slug>/ path.
    _ROOT         = "/kaggle/input/datasets/akshaym1104/dataset2"
    RAF_DB_DIR    = "/kaggle/input/datasets/prajwalm2213/raf-db/RAF-DB"   
    FER2013_DIR   = "/kaggle/input/datasets/prajwalm2213/fer-2013"              
    JAFFE_DIR     = "/kaggle/input/datasets/prajwalm2213/dataset2/jaffe/jaffe"                  # walked recursively, nested jaffe/jaffe is fine
    CKP_DIR       = f"{_ROOT}/CK"                      # -> anger/contempt/disgust/fear/happy/sadness/surprise (no "neutral")
    AFFECTNET_DIR = "/kaggle/input/datasets/mstjebashazida/affectnet"   # -> .../Test, .../Train, labels.csv

    # ── Output ────────────────────────────────────────────────────────────────
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    LOG_DIR        = "/kaggle/working/logs"
    EVAL_DIR       = "/kaggle/working/eval_results"

    # ── Model ─────────────────────────────────────────────────────────────────
    IMAGE_SIZE     = 224
    NUM_CLASSES    = 11          # compound emotions only
    EMBED_DIM      = 512
    LSTM_HIDDEN    = 256
    LSTM_LAYERS    = 2
    DROPOUT        = 0.3
    VIT_MODEL_NAME = "vit_base_patch16_224"

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE     = 32
    LEARNING_RATE  = 2e-4
    WEIGHT_DECAY   = 1e-4
    PATIENCE       = 10
    SEED           = 42

    # ── Compound class definitions ────────────────────────────────────────────
    COMPOUND_CLASSES = [
        "Happily Surprised",   "Happily Disgusted",  "Sadly Fearful",
        "Sadly Angry",         "Sadly Surprised",    "Sadly Disgusted",
        "Fearfully Angry",     "Fearfully Surprised","Angrily Surprised",
        "Angrily Disgusted",   "Disgustedly Surprised",
    ]
    BASIC_CLASSES = ["Angry","Disgust","Fear","Happy","Sad","Surprise","Neutral"]

    # compound_idx -> (basic_emotion_1, basic_emotion_2)
    # indices align to BASIC_CLASSES above
    COMPOUND_TO_BASIC = {
        0:(3,5), 1:(3,1), 2:(4,2),  3:(4,0), 4:(4,5),
        5:(4,1), 6:(2,0), 7:(2,5),  8:(0,5), 9:(0,1), 10:(1,5),
    }

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()
for d in [cfg.CHECKPOINT_DIR, cfg.LOG_DIR, cfg.EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(cfg.SEED)
print(f"Device : {cfg.DEVICE}")
print(f"RAF-DB exists    : {os.path.exists(cfg.RAF_DB_DIR)}")
print(f"FER-2013 exists  : {os.path.exists(cfg.FER2013_DIR)}")
print(f"JAFFE exists     : {os.path.exists(cfg.JAFFE_DIR)}")
print(f"CK+ exists       : {os.path.exists(cfg.CKP_DIR)}")
print(f"AffectNet exists : {os.path.exists(cfg.AFFECTNET_DIR)}")


## Data Augmentation / Transforms

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.Resize(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=20, p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=3, p=1.0),
            A.GaussNoise(p=1.0),
            A.MotionBlur(p=1.0),
        ], p=0.3),
        A.ColorJitter(brightness=0.3, contrast=0.3,
                      saturation=0.2, hue=0.1, p=0.5),
        A.CoarseDropout(num_holes_range=(1,8),
                        hole_height_range=(8,16),
                        hole_width_range=(8,16), p=0.3),
        A.Affine(translate_percent=0.1, scale=(0.85,1.15),
                 rotate=(-15,15), p=0.4),
        A.Normalize(mean=[0.485,0.456,0.406],
                    std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])
 
def get_val_transforms():
    return A.Compose([
        A.Resize(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406],
                    std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

## Dataset Loading (RAF-DB + FER-2013 + JAFFE + Morphological Splicing)

`SyntheticCompoundDataset` builds the 11 compound-emotion classes by
alpha-blending pairs of basic-emotion faces from FER-2013 (morphological
splicing). `get_dataloaders()` combines that with real RAF-DB images into
the training set, and returns `(train_loader, val_loader, test_loader)` —
RAF-DB's own held-out test split is reused for both val and test, matching
the original notebook's behavior (swap in a disjoint split later if you
want a stricter final evaluation).

In [ ]:
# ── Dataset classes ─────────────────────────────────────────────────────────
# ── Dataset classes ─────────────────────────────────────────────────────────
class RAFDBDataset(Dataset):
    # Name-based folders (any casing) AND RAF-DB's official numeric aligned-
    # label convention (1=Surprise,2=Fear,3=Disgust,4=Happiness,5=Sadness,
    # 6=Anger,7=Neutral) -- covers both common ways this dataset is shipped.
    FOLDER_TO_IDX = {
        "anger": 0, "angry": 0,
        "disgust": 1, "disgusted": 1,
        "fear": 2, "fearful": 2, "afraid": 2,
        "happiness": 3, "happy": 3,
        "sadness": 4, "sad": 4,
        "surprise": 5, "surprised": 5,
        "neutral": 6,
        "1": 5, "2": 2, "3": 1, "4": 3, "5": 4, "6": 0, "7": 6,
    }
    SPLIT_ALIASES = {
        "train": ["train", "Train", "TRAIN", "training"],
        "test":  ["test", "Test", "TEST", "testing", "valid", "val", "validation"],
    }

    def __init__(self, split="train", transform=None):
        self.transform = transform
        self.data_dir  = self._resolve_split_dir(split)
        self.samples   = []
        found_folders  = sorted(os.listdir(self.data_dir))
        matched = 0
        for folder in found_folders:
            label_idx = self.FOLDER_TO_IDX.get(folder.lower())
            if label_idx is None:
                continue
            matched += 1
            class_dir = os.path.join(self.data_dir, folder)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((os.path.join(class_dir, fname), label_idx))
        if matched == 0:
            print(f"  ⚠ RAFDBDataset[{split}]: 0/{len(found_folders)} subfolders "
                  f"under {self.data_dir} matched FOLDER_TO_IDX.\n"
                  f"    Found folders: {found_folders[:20]}\n"
                  f"    -> update FOLDER_TO_IDX (or RAF_DB_DIR) to match these names.")

    def _resolve_split_dir(self, split):
        base = cfg.RAF_DB_DIR
        if not os.path.isdir(base):
            raise FileNotFoundError(f"RAF_DB_DIR does not exist: {base}")
        for alias in self.SPLIT_ALIASES.get(split, [split]):
            candidate = os.path.join(base, alias)
            if os.path.isdir(candidate):
                return candidate
        raise FileNotFoundError(
            f"None of {self.SPLIT_ALIASES.get(split, [split])} found under {base}. "
            f"Contents of {base}: {os.listdir(base)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.long)


class FER2013Dataset(Dataset):
    def __init__(self, split="train", transform=None):
        self.transform = transform
        self.data_dir  = os.path.join(cfg.FER2013_DIR, split)
        self.samples   = []
        for label_idx, cls in enumerate(cfg.BASIC_CLASSES):
            class_dir = os.path.join(self.data_dir, cls.lower())
            if not os.path.exists(class_dir): continue
            for fname in os.listdir(class_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((os.path.join(class_dir, fname), label_idx))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        if self.transform: img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.long)


class JAFFEDataset(Dataset):
    EMOTION_MAP = {"HA":3,"SA":4,"AN":0,"FE":2,"DI":1,"SU":5,"NE":6}
    def __init__(self, transform=None):
        self.transform = transform
        self.samples   = []
        for root, _, files in os.walk(cfg.JAFFE_DIR):
            for fname in files:
                if not fname.lower().endswith((".tiff",".tif",".jpg",".png")): continue
                code = fname.split(".")[1][:2].upper()
                if code not in self.EMOTION_MAP: continue
                self.samples.append((os.path.join(root, fname), self.EMOTION_MAP[code]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        if img is None: img = np.array(Image.open(path).convert("RGB"))
        else: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform: img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.long)


class SyntheticCompoundDataset(Dataset):
    """Morphological splicing: alpha-blends pairs of basic-emotion FER-2013
    faces into synthetic examples of the 11 compound-emotion classes."""
    def __init__(self, fer_dataset, num_synthetic=8800, transform=None):
        self.transform = transform
        self.by_class  = {i: [] for i in range(7)}
        for path, label in fer_dataset.samples:
            self.by_class[label].append(path)
        self.samples = self._generate(num_synthetic)

    def _generate(self, n):
        samples, per_class = [], n // cfg.NUM_CLASSES
        for compound_idx, (e1, e2) in cfg.COMPOUND_TO_BASIC.items():
            pool1 = self.by_class.get(e1, [])
            pool2 = self.by_class.get(e2, [])
            if not pool1 or not pool2: continue
            for _ in range(per_class):
                p1 = pool1[np.random.randint(len(pool1))]
                p2 = pool2[np.random.randint(len(pool2))]
                samples.append((p1, p2, compound_idx))
        return samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        p1, p2, label = self.samples[idx]
        img1 = cv2.resize(cv2.cvtColor(cv2.imread(p1), cv2.COLOR_BGR2RGB),
                          (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))
        img2 = cv2.resize(cv2.cvtColor(cv2.imread(p2), cv2.COLOR_BGR2RGB),
                          (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))
        alpha = np.random.uniform(0.45, 0.55)
        mixed = (alpha * img1.astype(np.float32) +
                 (1 - alpha) * img2.astype(np.float32)).astype(np.uint8)
        if self.transform: mixed = self.transform(image=mixed)["image"]
        return mixed, torch.tensor(label, dtype=torch.long)


def make_weighted_sampler(dataset):
    label_list = []
    if hasattr(dataset, "datasets"):
        for ds in dataset.datasets:
            if isinstance(ds, SyntheticCompoundDataset):
                label_list += [s[2] for s in ds.samples]
            elif hasattr(ds, "samples"):
                label_list += [s[1] for s in ds.samples]
            elif hasattr(ds, "dataset"):
                base = ds.dataset
                label_list += [base.samples[i][1] for i in ds.indices]
    else:
        label_list = [s[1] for s in dataset.samples]

    if len(label_list) == 0:
        raise RuntimeError(
            "make_weighted_sampler: dataset is empty. "
            "Check RAF_DB_DIR / FER2013_DIR paths in Config.")

    n_bins  = max(label_list) + 1
    counts  = np.bincount(label_list, minlength=n_bins).astype(np.float32)
    weights = 1.0 / (counts + 1e-6)
    sw = torch.tensor([weights[l] for l in label_list], dtype=torch.float)
    return WeightedRandomSampler(sw, len(sw), replacement=True)


def get_dataloaders():
    """Returns (train_loader, val_loader, test_loader). RAF-DB's own test
    split is reused as both val and test — same behavior as the original
    single-model notebook this framework was built from."""
    train_tf = get_train_transforms()
    val_tf   = get_val_transforms()

    rafdb_train = RAFDBDataset(split="train", transform=train_tf)
    rafdb_val   = RAFDBDataset(split="test",  transform=val_tf)
    fer_raw     = FER2013Dataset(split="train", transform=None)

    synthetic = SyntheticCompoundDataset(fer_raw, num_synthetic=8800, transform=train_tf)
    combined  = ConcatDataset([rafdb_train, synthetic])
    sampler   = make_weighted_sampler(combined)

    print(f"  RAF-DB train   : {len(rafdb_train)}")
    print(f"  Synthetic      : {len(synthetic)}")
    print(f"  Combined total : {len(combined)}")
    print(f"  Val / Test     : {len(rafdb_val)}")

    train_loader = DataLoader(combined, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                              num_workers=4, pin_memory=True)
    val_loader   = DataLoader(rafdb_val, batch_size=cfg.BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    test_loader  = val_loader  # no disjoint held-out split upstream

    return train_loader, val_loader, test_loader


class FER2013Pool:
    """Minimal FER2013Pool for Stage 8 (dataset-wise evaluation) — exposes
    a `.by_class` dict of FER-2013 image paths grouped by basic emotion."""
    def __init__(self, split="train"):
        ds = FER2013Dataset(split=split, transform=None)
        self.by_class = {i: [] for i in range(7)}
        for path, label in ds.samples:
            self.by_class[label].append(path)


print("Dataset loading code ready \u2713")


## Stage 1 -- Model Architectures\n\nAll 7 architectures: shared attention primitives, SA-CNN, ALSTM, ViT encoder (reused unchanged), new backbones (ConvNeXt / EfficientNetV2 / Mamba-lite), generic N-branch fusion, and the model registry.

In [ ]:
"""
common_blocks.py
=================
REUSED, UNCHANGED components from your original notebook.

These are copy-pasted verbatim from your `CompoundEmotionModel` cell —
nothing here was modified. They are extracted into a mod
ule so every
new architecture (ConvNeXt+ViT, EfficientNetV2+ViT, ConvNeXt+Mamba, etc.)
can import and reuse SA-CNN, ALSTM, and the ViT encoder without
duplicating code.

Do NOT edit this file when adding new architectures — new backbones go
in `backbones_new.py` instead, so your original, already-validated
(64.34% val acc) building blocks stay byte-for-byte identical.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


# ─────────────────────────────────────────────────────────────────────────
# Attention primitives used inside SA-CNN
# ─────────────────────────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.shape
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx = self.fc(self.max_pool(x).view(b, c))
        return x * self.sigmoid(avg + mx).view(b, c, 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))


class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.cbam = CBAMBlock(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.cbam(out)
        return self.relu(out + self.shortcut(x))


# ─────────────────────────────────────────────────────────────────────────
# Branch 1 — SA-CNN (Self-Attention CNN)
# ─────────────────────────────────────────────────────────────────────────
class SACNN(nn.Module):
    def __init__(self, embed_dim=512, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.stage1 = self._make_stage(64, 128, stride=1, blocks=2)
        self.stage2 = self._make_stage(128, 256, stride=2, blocks=2)
        self.stage3 = self._make_stage(256, 512, stride=2, blocks=2)
        self.stage4 = self._make_stage(512, 512, stride=2, blocks=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(512, embed_dim)
        self.bn_out = nn.BatchNorm1d(embed_dim)
        self._init_weights()

    def _make_stage(self, in_ch, out_ch, stride, blocks):
        layers = [ResidualBlock(in_ch, out_ch, stride=stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.bn_out(self.fc(x))


# ─────────────────────────────────────────────────────────────────────────
# Branch 2 — ALSTM (Attention-based LSTM over flattened patches)
# ─────────────────────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_ch=3, patch_dim=256):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_ch, patch_dim, kernel_size=patch_size, stride=patch_size),
            nn.Flatten(2),
        )

    def forward(self, x):
        return self.proj(x).transpose(1, 2)


class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False),
        )

    def forward(self, lstm_out):
        scores = self.attn(lstm_out)
        weights = F.softmax(scores, dim=1)
        return (weights * lstm_out).sum(dim=1), weights.squeeze(-1)


class ALSTM(nn.Module):
    def __init__(self, img_size=224, patch_size=16, patch_dim=256,
                 hidden_dim=256, num_layers=2, embed_dim=512, dropout=0.3):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, patch_dim=patch_dim)
        self.lstm = nn.LSTM(patch_dim, hidden_dim, num_layers, batch_first=True,
                             bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.attention = TemporalAttention(hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, embed_dim)
        self.bn_out = nn.BatchNorm1d(embed_dim)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
        nn.init.xavier_normal_(self.fc.weight)

    def forward(self, x):
        patches = self.patch_embed(x)
        lstm_out, _ = self.lstm(patches)
        context, _ = self.attention(lstm_out)
        return self.bn_out(self.fc(self.drop(context)))


# ─────────────────────────────────────────────────────────────────────────
# Branch 3 — ViT encoder (timm backbone + projection head)
# This same "freeze / unfreeze_last_n_blocks / unfreeze_all" interface is
# the contract every NEW backbone in backbones_new.py must also implement,
# so the 3-stage curriculum (run_stage 1/2/3) works unmodified for every
# architecture.
# ─────────────────────────────────────────────────────────────────────────
class ViTEncoder(nn.Module):
    def __init__(self, model_name="vit_base_patch16_224", embed_dim=512,
                 dropout=0.3, pretrained=True):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        vit_dim = self.vit.num_features
        self.projection = nn.Sequential(
            nn.LayerNorm(vit_dim),
            nn.Dropout(dropout),
            nn.Linear(vit_dim, embed_dim),
            nn.GELU(),
            nn.BatchNorm1d(embed_dim),
        )
        self._freeze_all()

    def _freeze_all(self):
        for p in self.vit.parameters():
            p.requires_grad = False

    def unfreeze_last_n_blocks(self, n=8):
        for p in self.vit.norm.parameters():
            p.requires_grad = True
        for block in list(self.vit.blocks)[-n:]:
            for p in block.parameters():
                p.requires_grad = True
        tr = sum(p.numel() for p in self.vit.parameters() if p.requires_grad)
        print(f"  ViT unfrozen last {n} blocks — trainable ViT params: {tr:,}")

    def unfreeze_all(self):
        for p in self.vit.parameters():
            p.requires_grad = True
        print("  ViT fully unfrozen")

    def forward(self, x):
        return self.projection(self.vit(x))

In [ ]:
"""
backbones_new.py
=================
NEW encoder branches required for the 6 baseline architectures. Each one
implements the SAME interface as `ViTEncoder` in common_blocks.py:

    forward(x)                  -> (B, embed_dim) feature vector
    unfreeze_last_n_blocks(n)   -> stage-2 fine-tuning (partial unfreeze)
    unfreeze_all()              -> stage-3 fine-tuning (full unfreeze)

Matching this interface is what lets the SAME 3-stage training curriculum
(`run_stage` stage1/2/3 in your notebook) drive every architecture without
any changes to the training loop — which is the "fair comparison"
requirement in your spec.

Backbones added here:
    - ConvNeXtEncoder          (for ConvNeXt+ViT, ConvNeXt+Mamba)
    - EfficientNetV2Encoder    (for EfficientNetV2+ViT)
    - MambaLiteEncoder         (for ConvNeXt+Mamba)

Mamba compatibility note
-------------------------
The official `mamba-ssm` package ships fused CUDA kernels
(selective-scan, causal-conv1d) built for a specific set of GPU compute
capabilities. Pascal-generation cards — including your Tesla P100
(sm_60) — commonly fall outside that supported set and either fail to
build the package or crash at kernel-launch time.

To keep "ConvNeXt+Mamba" actually runnable on your Kaggle P100, this
file implements `MambaLiteEncoder`: a pure-PyTorch selective state-space
block (same core recurrence as Mamba's S6 block: input-dependent A/B/C,
sequential scan over the patch sequence) with NO custom CUDA kernel. It
is slower per step than the fused kernel but numerically follows the
same selective-SSM formulation, so the architecture comparison stays
meaningful.

If you later run this on a Turing/Ampere+ GPU and want the real fused
kernel instead, replace `MambaLiteEncoder`'s `SelectiveScanLite` block
with `from mamba_ssm import Mamba` — the surrounding encoder class
(patch embedding, projection head, freeze/unfreeze contract) stays
identical either way.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


# ─────────────────────────────────────────────────────────────────────────
# Shared base: wraps a timm CNN backbone with the same projection head
# style as ViTEncoder, plus a generic freeze/unfreeze-by-stage mechanism.
# ─────────────────────────────────────────────────────────────────────────
class _TimmCNNEncoder(nn.Module):
    def __init__(self, model_name, embed_dim=512, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, global_pool="avg"
        )
        feat_dim = self.backbone.num_features
        self.projection = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, embed_dim),
            nn.GELU(),
            nn.BatchNorm1d(embed_dim),
        )
        self._freeze_all()

    # ---- override in subclasses: return blocks ordered shallow -> deep ----
    def _ordered_blocks(self):
        raise NotImplementedError

    def _freeze_all(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_n_blocks(self, n=8):
        blocks = self._ordered_blocks()
        for block in blocks[-n:]:
            for p in block.parameters():
                p.requires_grad = True
        # keep final norm trainable too, mirroring ViTEncoder behaviour
        if hasattr(self.backbone, "norm_pre"):
            for p in self.backbone.norm_pre.parameters():
                p.requires_grad = True
        if hasattr(self.backbone, "head_hidden_size"):
            pass  # no-op, placeholder for future timm versions
        tr = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        print(f"  {self.__class__.__name__} unfrozen last {n} blocks "
              f"— trainable params: {tr:,}")

    def unfreeze_all(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
        print(f"  {self.__class__.__name__} fully unfrozen")

    def forward(self, x):
        feats = self.backbone(x)          # (B, feat_dim), already pooled
        return self.projection(feats)


class ConvNeXtEncoder(_TimmCNNEncoder):
    """ConvNeXt backbone (timm). Default: convnext_tiny (28M params, ImageNet
    pretrained) — closest capacity match to vit_base among small ConvNeXt
    variants, keeping the comparison reasonably fair on parameter count."""

    def __init__(self, model_name="convnext_tiny", embed_dim=512,
                 dropout=0.3, pretrained=True):
        super().__init__(model_name, embed_dim, dropout, pretrained)

    def _ordered_blocks(self):
        # timm ConvNeXt: backbone.stages -> ModuleList of 4 stages,
        # each stage.blocks -> ModuleList of ConvNeXtBlock
        blocks = []
        for stage in self.backbone.stages:
            blocks.extend(list(stage.blocks))
        return blocks


class EfficientNetV2Encoder(_TimmCNNEncoder):
    """EfficientNetV2 backbone (timm). Default: tf_efficientnetv2_s."""

    def __init__(self, model_name="tf_efficientnetv2_s", embed_dim=512,
                 dropout=0.3, pretrained=True):
        super().__init__(model_name, embed_dim, dropout, pretrained)

    def _ordered_blocks(self):
        # timm EfficientNetV2: backbone.blocks -> Sequential of stages,
        # each stage -> Sequential of individual MBConv/FusedMBConv blocks
        blocks = []
        for stage in self.backbone.blocks:
            blocks.extend(list(stage))
        return blocks


# ─────────────────────────────────────────────────────────────────────────
# Mamba-lite: pure-PyTorch selective state-space block (P100-safe)
# ─────────────────────────────────────────────────────────────────────────
class SelectiveScanLite(nn.Module):
    """
    Minimal selective SSM (Mamba S6-style) block, no custom CUDA kernel.

    For each channel, maintains a hidden state h_t updated as:
        h_t = exp(delta_t * A) * h_{t-1} + delta_t * B_t * x_t
        y_t = C_t . h_t
    where delta, B, C are input-dependent (the "selective" part of Mamba).
    Implemented as an explicit sequential scan over the patch sequence —
    correct, GPU-portable, but O(seq_len) sequential steps (no parallel
    scan / fused kernel).
    """

    def __init__(self, d_model, d_state=16, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = expand * d_model
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, self.d_inner * 2)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=3,
                                 padding=1, groups=self.d_inner)
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1)  # -> B, C, delta
        self.dt_proj = nn.Linear(1, self.d_inner)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))          # (d_inner, d_state)
        self.D = nn.Parameter(torch.ones(self.d_inner))

        self.out_proj = nn.Linear(self.d_inner, d_model)

    def forward(self, x):
        # x: (B, L, d_model)
        B_, L, _ = x.shape
        xz = self.in_proj(x)                             # (B, L, 2*d_inner)
        x_in, z = xz.chunk(2, dim=-1)

        x_in = self.conv1d(x_in.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_in = F.silu(x_in)                               # (B, L, d_inner)

        proj = self.x_proj(x_in)                          # (B, L, 2*d_state+1)
        Bs, Cs, dt = torch.split(proj, [self.d_state, self.d_state, 1], dim=-1)
        delta = F.softplus(self.dt_proj(dt))               # (B, L, d_inner)

        A = -torch.exp(self.A_log)                        # (d_inner, d_state)
        h = torch.zeros(B_, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dA = torch.exp(delta[:, t].unsqueeze(-1) * A)              # (B, d_inner, d_state)
            dB = delta[:, t].unsqueeze(-1) * Bs[:, t].unsqueeze(1)     # (B, d_inner, d_state)
            h = dA * h + dB * x_in[:, t].unsqueeze(-1)
            y_t = (h * Cs[:, t].unsqueeze(1)).sum(-1)                  # (B, d_inner)
            ys.append(y_t)
        y = torch.stack(ys, dim=1)                          # (B, L, d_inner)
        y = y + x_in * self.D
        y = y * F.silu(z)
        return self.out_proj(y)


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.ssm = SelectiveScanLite(d_model, d_state=d_state)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.drop(self.ssm(self.norm(x)))


class MambaLiteEncoder(nn.Module):
    """
    Patchifies the image (same 16x16 patch stem as ALSTM/ViT) and runs a
    stack of pure-PyTorch selective-SSM blocks over the resulting
    sequence, then mean-pools to a single embedding. Matches the
    freeze/unfreeze contract of the other encoders (all layers are
    trained from scratch, so "unfreeze" is a no-op — included only so
    the shared stage1/2/3 curriculum code can call it uniformly).
    """

    def __init__(self, img_size=224, patch_size=16, d_model=256,
                 d_state=16, depth=4, embed_dim=512, dropout=0.3):
        super().__init__()
        self.patch_embed = nn.Sequential(
            nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size),
            nn.Flatten(2),
        )
        n_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList([
            MambaBlock(d_model, d_state=d_state, dropout=dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.projection = nn.Sequential(
            nn.Linear(d_model, embed_dim),
            nn.GELU(),
            nn.BatchNorm1d(embed_dim),
        )

    # trained end-to-end from scratch; kept only for interface parity
    def unfreeze_last_n_blocks(self, n=8):
        print("  MambaLiteEncoder trains from scratch — unfreeze is a no-op")

    def unfreeze_all(self):
        print("  MambaLiteEncoder trains from scratch — unfreeze is a no-op")

    def forward(self, x):
        tokens = self.patch_embed(x).transpose(1, 2)      # (B, N, d_model)
        tokens = tokens + self.pos_embed
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens).mean(dim=1)             # (B, d_model)
        return self.projection(tokens)

In [ ]:
"""
fusion.py
=========
Generalizes your original 3-branch `AttentionFusion` (SA-CNN + ALSTM + ViT)
to N branches, so the same gated-attention fusion mechanism can combine
2 branches (e.g. ConvNeXt + ViT) or 3 branches (the proposed model)
without duplicating code.

For N=3 with embed_dim=512, this is mathematically identical to your
original `AttentionFusion` class — same gate architecture
(Linear -> ReLU -> Linear -> Softmax), just written to accept a list.
"""
import torch
import torch.nn as nn


class GenericAttentionFusion(nn.Module):
    def __init__(self, embed_dim=512, num_branches=3):
        super().__init__()
        self.num_branches = num_branches
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * num_branches, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_branches),
            nn.Softmax(dim=-1),
        )

    def forward(self, feats):
        # feats: list of (B, embed_dim) tensors, len == num_branches
        combined = torch.cat(feats, dim=-1)
        weights = self.gate(combined)                      # (B, num_branches)
        fused = sum(weights[:, i:i + 1] * feats[i] for i in range(self.num_branches))
        return fused, weights

In [ ]:
"""
models.py
=========
All 7 architectures required for the comparison study, built on a single
shared base class so training/eval/benchmarking code never needs to
special-case any of them.

    1. SACNN_ALSTM          — SA-CNN + ALSTM
    2. SACNN_ViT            — SA-CNN + ViT
    3. ConvNeXt_ViT         — ConvNeXt + ViT
    4. EfficientNetV2_ViT   — EfficientNetV2 + ViT
    5. ConvNeXt_Mamba       — ConvNeXt + Mamba (pure-PyTorch, P100-safe)
    6. ALSTM_ViT            — ALSTM + ViT
    7. ProposedModel        — Morphological Splicing (data-level) +
                              SA-CNN + ALSTM + ViT  (== your original
                              CompoundEmotionModel, kept byte-for-byte
                              identical for exact reproducibility of the
                              64.34% result)

Design
------
`BaseFusionModel` takes an ordered dict of {name: encoder_module} branches
and:
  - runs every branch on the same input image
  - fuses their embeddings with `GenericAttentionFusion`
  - classifies through the SAME classifier head shape as your original
    proposed model (Linear->LN->GELU->Dropout ->  Linear->LN->GELU->Dropout
    -> Linear)
  - returns (logits, fusion_weights) — identical signature to your
    original `CompoundEmotionModel.forward`, so `train_one_epoch` /
    `validate` in your notebook work UNCHANGED for every architecture.

`stage1_setup / stage2_setup(n) / stage3_setup` iterate over branches and
call `unfreeze_last_n_blocks` / `unfreeze_all` on any branch that defines
them (ViT, ConvNeXt, EfficientNetV2). Branches trained from scratch
(SA-CNN, ALSTM, Mamba-lite) are always fully trainable and simply ignore
the freeze calls — so the exact same 3-stage curriculum
(`run_stage(..., stage=1/2/3)`) drives every model with zero changes.
"""
from collections import OrderedDict
import torch
import torch.nn as nn




def _make_classifier(embed_dim, num_classes, dropout):
    clf = nn.Sequential(
        nn.Linear(embed_dim, 512),
        nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(512, 256),
        nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout * 0.5),
        nn.Linear(256, num_classes),
    )
    for m in clf.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)
    return clf


class BaseFusionModel(nn.Module):
    def __init__(self, branches: "OrderedDict[str, nn.Module]",
                 num_classes=11, embed_dim=512, dropout=0.3):
        super().__init__()
        self.branch_names = list(branches.keys())
        self.branches = nn.ModuleDict(branches)
        self.fusion = GenericAttentionFusion(embed_dim=embed_dim,
                                              num_branches=len(branches))
        self.classifier = _make_classifier(embed_dim, num_classes, dropout)

    def _trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def stage1_setup(self):
        """Freeze any pretrained branch entirely; scratch branches stay trainable."""
        for branch in self.branches.values():
            if hasattr(branch, "_freeze_all"):
                branch._freeze_all()
        print(f"  Stage 1 trainable params: {self._trainable_params():,}")

    def stage2_setup(self, n=8):
        """Unfreeze last n blocks of every pretrained branch."""
        for branch in self.branches.values():
            if hasattr(branch, "unfreeze_last_n_blocks"):
                branch.unfreeze_last_n_blocks(n)
        print(f"  Stage 2 trainable params: {self._trainable_params():,}")

    def stage3_setup(self):
        """Fully unfreeze every pretrained branch."""
        for branch in self.branches.values():
            if hasattr(branch, "unfreeze_all"):
                branch.unfreeze_all()
        print(f"  Stage 3 trainable params: {self._trainable_params():,}")

    def forward(self, x):
        feats = [self.branches[name](x) for name in self.branch_names]
        fused, weights = self.fusion(feats)
        return self.classifier(fused), weights


# ─────────────────────────────────────────────────────────────────────────
# 1. SA-CNN + ALSTM  (both trained from scratch — no freeze stages needed,
#    but stage1/2/3 calls are still safe no-ops so run_stage() works as-is)
# ─────────────────────────────────────────────────────────────────────────
def build_sacnn_alstm(cfg):
    branches = OrderedDict([
        ("sacnn", SACNN(embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT)),
        ("alstm", ALSTM(hidden_dim=cfg.LSTM_HIDDEN, num_layers=cfg.LSTM_LAYERS,
                         embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT)),
    ])
    return BaseFusionModel(branches, cfg.NUM_CLASSES, cfg.EMBED_DIM, cfg.DROPOUT)


# ─────────────────────────────────────────────────────────────────────────
# 2. SA-CNN + ViT
# ─────────────────────────────────────────────────────────────────────────
def build_sacnn_vit(cfg):
    branches = OrderedDict([
        ("sacnn", SACNN(embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT)),
        ("vit", ViTEncoder(model_name=cfg.VIT_MODEL_NAME, embed_dim=cfg.EMBED_DIM,
                            dropout=cfg.DROPOUT, pretrained=True)),
    ])
    return BaseFusionModel(branches, cfg.NUM_CLASSES, cfg.EMBED_DIM, cfg.DROPOUT)


# ─────────────────────────────────────────────────────────────────────────
# 3. ConvNeXt + ViT
# ─────────────────────────────────────────────────────────────────────────
def build_convnext_vit(cfg):
    branches = OrderedDict([
        ("convnext", ConvNeXtEncoder(embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT,
                                      pretrained=True)),
        ("vit", ViTEncoder(model_name=cfg.VIT_MODEL_NAME, embed_dim=cfg.EMBED_DIM,
                            dropout=cfg.DROPOUT, pretrained=True)),
    ])
    return BaseFusionModel(branches, cfg.NUM_CLASSES, cfg.EMBED_DIM, cfg.DROPOUT)


# ─────────────────────────────────────────────────────────────────────────
# 4. EfficientNetV2 + ViT
# ─────────────────────────────────────────────────────────────────────────
def build_effnetv2_vit(cfg):
    branches = OrderedDict([
        ("effnetv2", EfficientNetV2Encoder(embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT,
                                            pretrained=True)),
        ("vit", ViTEncoder(model_name=cfg.VIT_MODEL_NAME, embed_dim=cfg.EMBED_DIM,
                            dropout=cfg.DROPOUT, pretrained=True)),
    ])
    return BaseFusionModel(branches, cfg.NUM_CLASSES, cfg.EMBED_DIM, cfg.DROPOUT)


# ─────────────────────────────────────────────────────────────────────────
# 5. ConvNeXt + Mamba (pure-PyTorch selective-SSM, P100-safe)
# ─────────────────────────────────────────────────────────────────────────
def build_convnext_mamba(cfg):
    branches = OrderedDict([
        ("convnext", ConvNeXtEncoder(embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT,
                                      pretrained=True)),
        ("mamba", MambaLiteEncoder(img_size=cfg.IMAGE_SIZE, embed_dim=cfg.EMBED_DIM,
                                    dropout=cfg.DROPOUT)),
    ])
    return BaseFusionModel(branches, cfg.NUM_CLASSES, cfg.EMBED_DIM, cfg.DROPOUT)


# ─────────────────────────────────────────────────────────────────────────
# 6. ALSTM + ViT
# ─────────────────────────────────────────────────────────────────────────
def build_alstm_vit(cfg):
    branches = OrderedDict([
        ("alstm", ALSTM(hidden_dim=cfg.LSTM_HIDDEN, num_layers=cfg.LSTM_LAYERS,
                         embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT)),
        ("vit", ViTEncoder(model_name=cfg.VIT_MODEL_NAME, embed_dim=cfg.EMBED_DIM,
                            dropout=cfg.DROPOUT, pretrained=True)),
    ])
    return BaseFusionModel(branches, cfg.NUM_CLASSES, cfg.EMBED_DIM, cfg.DROPOUT)


# ─────────────────────────────────────────────────────────────────────────
# 7. Proposed model — kept EXACTLY as your original CompoundEmotionModel
#    (3-way fusion, SA-CNN + ALSTM + ViT) so its results are reproducible
#    byte-for-byte. Expressed here via BaseFusionModel for interface
#    consistency with the other 6, but numerically identical: same three
#    branches, same GenericAttentionFusion(embed_dim, num_branches=3)
#    (== your original AttentionFusion for N=3), same classifier head.
# ─────────────────────────────────────────────────────────────────────────
def build_proposed(cfg):
    branches = OrderedDict([
        ("sacnn", SACNN(embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT)),
        ("alstm", ALSTM(hidden_dim=cfg.LSTM_HIDDEN, num_layers=cfg.LSTM_LAYERS,
                         embed_dim=cfg.EMBED_DIM, dropout=cfg.DROPOUT)),
        ("vit", ViTEncoder(model_name=cfg.VIT_MODEL_NAME, embed_dim=cfg.EMBED_DIM,
                            dropout=cfg.DROPOUT, pretrained=True)),
    ])
    return BaseFusionModel(branches, cfg.NUM_CLASSES, cfg.EMBED_DIM, cfg.DROPOUT)


# ─────────────────────────────────────────────────────────────────────────
# Registry — this is what lets you write, in a later stage:
#
#     for name in MODEL_REGISTRY:
#         model = MODEL_REGISTRY[name](cfg).to(cfg.DEVICE)
#         train(model); evaluate(model); benchmark(model)
# ─────────────────────────────────────────────────────────────────────────
MODEL_REGISTRY = {
    "SACNN_ALSTM":        build_sacnn_alstm,
    "SACNN_ViT":          build_sacnn_vit,
    "ConvNeXt_ViT":       build_convnext_vit,
    "EfficientNetV2_ViT": build_effnetv2_vit,
    "ConvNeXt_Mamba":     build_convnext_mamba,
    "ALSTM_ViT":          build_alstm_vit,
    "Proposed":           build_proposed,
}


def build_model(name, cfg):
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{name}'. Choices: {list(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[name](cfg).to(cfg.DEVICE)

## Training Utilities (reused from your original notebook -- one small patch applied)

`mixup_batch`, `compute_metrics`, `train_one_epoch`, `validate`, `save_ckpt`,
`load_ckpt`, `run_stage` are unchanged **except** the per-epoch log entry inside
`run_stage()` now also stores `tr_loss`, `vl_loss`, and `elapsed_sec` -- needed
for Stage 4's loss curves and Stage 5's training-time benchmark. Everything else
is byte-for-byte identical to your original.


In [ ]:
def mixup_batch(imgs, labels, num_classes, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    y_a = F.one_hot(labels, num_classes).float()
    y_b = F.one_hot(labels[idx], num_classes).float()
    return mixed, lam * y_a + (1 - lam) * y_b
 
def compute_metrics(preds, labels):
    p, l = np.array(preds), np.array(labels)
    acc = (p == l).mean() * 100
    f1  = f1_score(l, p, average="weighted", zero_division=0) * 100
    return acc, f1
 
def train_one_epoch(model, loader, optimizer, scheduler,
                    scaler, criterion, epoch, total):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
 
    for imgs, labels in loader:
        imgs   = imgs.to(cfg.DEVICE, non_blocking=True)
        labels = labels.to(cfg.DEVICE, non_blocking=True)
 
        use_mixup = random.random() < 0.4
        if use_mixup:
            imgs, soft_labels = mixup_batch(imgs, labels, cfg.NUM_CLASSES)
 
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            logits, _ = model(imgs)
            if use_mixup:
                lp   = F.log_softmax(logits, dim=-1)
                loss = -(soft_labels * lp).sum(dim=-1).mean()
            else:
                loss = criterion(logits, labels)
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
 
        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
 
    acc, f1 = compute_metrics(all_preds, all_labels)
    return total_loss / len(loader), acc, f1
 
@torch.no_grad()
def validate(model, loader, criterion, epoch, total):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, fw_log = [], [], []
 
    for imgs, labels in loader:
        imgs   = imgs.to(cfg.DEVICE, non_blocking=True)
        labels = labels.to(cfg.DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits, weights = model(imgs)
            loss = criterion(logits, labels)
        total_loss  += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        fw_log.append(weights.cpu().numpy())
 
    fw = np.concatenate(fw_log, axis=0).mean(axis=0)
    acc, f1 = compute_metrics(all_preds, all_labels)
    return total_loss / len(loader), acc, f1, fw
 
def save_ckpt(state, fname):
    torch.save(state, os.path.join(cfg.CHECKPOINT_DIR, fname))
    print(f"    Saved → {fname}")
 
def load_ckpt(model, fname):
    path = os.path.join(cfg.CHECKPOINT_DIR, fname)
    if not os.path.exists(path):
        print(f"  ⚠  {fname} not found")
        return 0, 0.0
    ckpt = torch.load(path, map_location=cfg.DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    print(f"  Loaded {fname} — compound_acc={ckpt['best_acc']:.2f}%")
    return ckpt["epoch"], ckpt["best_acc"]
 
def run_stage(model, train_loader, val_loader,
              stage, epochs, lr, ckpt_name):
    """
    Single training stage.
    Val loader is ALWAYS the compound val split — early stopping
    is based on compound emotion accuracy.
    """
    # Class-balanced CE with label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=cfg.WEIGHT_DECAY
    )
    scheduler = OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=len(train_loader),
        epochs=epochs, pct_start=0.3,
        anneal_strategy="cos",
        div_factor=10, final_div_factor=100
    )
    scaler      = torch.amp.GradScaler("cuda")
    best_acc    = 0.0
    patience_ct = 0
    log         = []
 
    print(f"\n{'='*65}")
    print(f"  Stage {stage} — {epochs} epochs | max_lr={lr:.2e}")
    print(f"  Val = compound emotion split (this is the correct metric)")
    print(f"{'='*65}")
 
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = train_one_epoch(
            model, train_loader, optimizer, scheduler,
            scaler, criterion, epoch, epochs)
        vl_loss, vl_acc, vl_f1, fw = validate(
            model, val_loader, criterion, epoch, epochs)
 
        elapsed = time.time() - t0
        fw_str = ",".join(f"{w:.2f}" for w in fw)
        print(f"  E{epoch:02d}/{epochs} | "
              f"tr_acc={tr_acc:.1f}% f1={tr_f1:.1f}% | "
              f"val_acc={vl_acc:.1f}% f1={vl_f1:.1f}% | "
              f"fw=[{fw_str}] | "
              f"{elapsed:.0f}s")
 
        log.append({"epoch": epoch, "stage": stage,
                    "tr_loss": tr_loss, "vl_loss": vl_loss,
                    "tr_acc": tr_acc, "vl_acc": vl_acc, "vl_f1": vl_f1,
                    "elapsed_sec": elapsed})
 
        if vl_acc > best_acc:
            best_acc    = vl_acc
            patience_ct = 0
            save_ckpt({"epoch": epoch, "model": model.state_dict(),
                       "optimizer": optimizer.state_dict(),
                       "best_acc": best_acc,
                       "stage": stage}, ckpt_name)
        else:
            patience_ct += 1
            if patience_ct >= cfg.PATIENCE:
                print(f"  Early stop at epoch {epoch} "
                      f"(no improvement for {cfg.PATIENCE} epochs)")
                break
 
    print(f"\n  Stage {stage} best compound accuracy: {best_acc:.2f}%")
    return log, best_acc
 
print("Training utilities loaded ✓")


## Stage 2 -- Generic Training Pipeline\n\nRuns the same 3-stage curriculum (frozen -> partial unfreeze -> full unfreeze) with identical epochs/LR/optimizer/scheduler for every architecture.

In [ ]:
"""
train_pipeline.py — Stage 2
============================
Generic `train_model(name, train_loader, val_loader, cfg)` that runs the
SAME 3-stage curriculum — same epoch counts, same LR schedule, same
optimizer/scheduler/patience — on any of the 7 architectures in
MODEL_REGISTRY. This is what your spec calls for: "EXACT same ...
epochs, batch size, learning rate ... to ensure a fair comparison."

Nothing about your existing training internals changes:
    - run_stage(), train_one_epoch(), validate(), save_ckpt(), load_ckpt()
      are reused exactly as they already exist in your notebook.
    - The only new code is the orchestration: which model, which
      checkpoint names, and looping stage1 -> stage2 -> stage3.

UPDATED: checkpoints now persist per-STAGE, not just per-finished-model.
A 12h Kaggle session can die mid-model (e.g. partway through Stage 3 of
model 5). Previously that whole model's progress was lost because nothing
got zipped until all 3 stages finished. Now:
    - `on_stage_done(model_name, stage_num, cfg)` fires right after each
      stage's checkpoint is saved, so the driving loop can persist/zip
      after every stage instead of only at the very end.
    - `resume_stage` lets a restarted session skip stages that already
      finished (their checkpoints get reloaded instead of retrained),
      so you resume model 5 at Stage 3, not from Stage 1 again.

Paste this as its own cell AFTER the cell that defines run_stage /
train_one_epoch / validate / save_ckpt / load_ckpt, and AFTER the
models.py cell (build_model / MODEL_REGISTRY).
"""

def train_model(model_name, train_loader, val_loader, cfg,
                 stage_epochs=(25, 25, 10), resume_stage=1, on_stage_done=None):
    """
    Trains ONE architecture through the standard 3-stage curriculum:
        Stage 1 — pretrained branches fully frozen (if any)
        Stage 2 — last 8 blocks of pretrained branches unfrozen
        Stage 3 — pretrained branches fully unfrozen, low LR fine-tune

    For architectures with no pretrained branch (e.g. SACNN_ALSTM), the
    stageN_setup() calls are harmless no-ops (see models.py) — the model
    still goes through all 3 stages with the same epoch/LR schedule as
    every other model, preserving a fair, identical training budget
    across the whole comparison.

    resume_stage: 1, 2, or 3 — stages before this are SKIPPED and their
        checkpoints reloaded instead of retrained (used when resuming a
        model that got partway through in a previous session).
    on_stage_done: optional callback(model_name, stage_num, cfg), fired
        right after each stage's checkpoint is written to disk — hook a
        persist/zip call here so progress survives a mid-model session kill.

    Returns: (best_acc, ckpt_path, full_log)
    """
    print(f"\n{'#'*65}")
    print(f"#  TRAINING: {model_name}  (resuming from stage {resume_stage})" if resume_stage > 1
          else f"#  TRAINING: {model_name}")
    print(f"{'#'*65}")

    model = build_model(model_name, cfg)
    total_p = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {total_p:,}")

    e1, e2, e3 = stage_epochs
    full_log = []
    best1 = best2 = best3 = float("nan")

    # ── Stage 1 ──────────────────────────────────────────────────────────
    model.stage1_setup()
    if resume_stage <= 1:
        log1, best1 = run_stage(
            model, train_loader, val_loader,
            stage=1, epochs=e1, lr=cfg.LEARNING_RATE,
            ckpt_name=f"{model_name}_stage1_best.pth",
        )
        full_log += log1
    else:
        print(f"  ⏭  Stage 1 already completed in a previous session — loading checkpoint.")
    load_ckpt(model, f"{model_name}_stage1_best.pth")
    torch.cuda.empty_cache()
    if on_stage_done:
        on_stage_done(model_name, 1, cfg)

    # ── Stage 2 ──────────────────────────────────────────────────────────
    model.stage2_setup(n=8)
    if resume_stage <= 2:
        log2, best2 = run_stage(
            model, train_loader, val_loader,
            stage=2, epochs=e2, lr=cfg.LEARNING_RATE * 0.2,
            ckpt_name=f"{model_name}_stage2_best.pth",
        )
        full_log += log2
    else:
        print(f"  ⏭  Stage 2 already completed in a previous session — loading checkpoint.")
    load_ckpt(model, f"{model_name}_stage2_best.pth")
    torch.cuda.empty_cache()
    if on_stage_done:
        on_stage_done(model_name, 2, cfg)

    # ── Stage 3 ──────────────────────────────────────────────────────────
    model.stage3_setup()
    # Enable grad checkpointing on any branch that supports it (saves VRAM
    # during full fine-tune) — mirrors your original
    # `model.vit.vit.set_grad_checkpointing(True)` call, generalized so it
    # doesn't crash on architectures without a ViT branch.
    for branch in model.branches.values():
        inner = getattr(branch, "vit", None)          # ViTEncoder wraps timm model as .vit
        target = inner if inner is not None else getattr(branch, "backbone", None)
        if target is not None and hasattr(target, "set_grad_checkpointing"):
            target.set_grad_checkpointing(True)

    if resume_stage <= 3:
        log3, best3 = run_stage(
            model, train_loader, val_loader,
            stage=3, epochs=e3, lr=cfg.LEARNING_RATE * 0.02,
            ckpt_name=f"{model_name}_stage3_best.pth",
        )
        full_log += log3
    else:
        print(f"  ⏭  Stage 3 already completed in a previous session — loading checkpoint.")
        load_ckpt(model, f"{model_name}_stage3_best.pth")
    if on_stage_done:
        on_stage_done(model_name, 3, cfg)

    # ── Save per-model training log ─────────────────────────────────────
    log_path = os.path.join(cfg.LOG_DIR, f"{model_name}_training_log.json")
    with open(log_path, "w") as f:
        json.dump(full_log, f, indent=2)

    ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, f"{model_name}_stage3_best.pth")
    print(f"\n{'='*65}")
    print(f"  {model_name} TRAINING COMPLETE")
    print(f"  Stage 1 best: {best1:.2f}%  |  Stage 2 best: {best2:.2f}%  "
          f"|  Stage 3 best: {best3:.2f}%")
    print(f"  Checkpoint → {ckpt_path}")
    print(f"{'='*65}")

    return best3, ckpt_path, full_log


def train_all_models(MODEL_LIST, train_loader, val_loader, cfg,
                      stage_epochs=(25, 25, 10)):
    """
    Runs train_model() for every name in MODEL_LIST and returns a summary
    dict: {model_name: {"best_acc": ..., "ckpt_path": ...}}.

    Saves the summary to cfg.LOG_DIR/all_models_summary.json as it goes,
    so a Kaggle session timeout partway through doesn't lose earlier
    results.
    """
    summary = {}
    summary_path = os.path.join(cfg.LOG_DIR, "all_models_summary.json")

    for name in MODEL_LIST:
        best_acc, ckpt_path, _ = train_model(
            name, train_loader, val_loader, cfg, stage_epochs=stage_epochs
        )
        summary[name] = {"best_acc": best_acc, "ckpt_path": ckpt_path}
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)

    print(f"\n\n{'='*65}")
    print(f"  ALL MODELS TRAINED")
    print(f"{'='*65}")
    for name, r in summary.items():
        print(f"  {name:20s} best_acc={r['best_acc']:.2f}%")
    print(f"\n  Summary → {summary_path}")
    return summary


# ─────────────────────────────────────────────────────────────────────────
# Example usage (uncomment and run once your dataloaders are ready):
#
# MODEL_LIST = list(MODEL_REGISTRY.keys())   # all 7, or pick a subset
# summary = train_all_models(MODEL_LIST, train_loader, val_loader, cfg)
# ─────────────────────────────────────────────────────────────────────────


## Stage 3 -- Generic Evaluation Pipeline\n\nAccuracy, precision, recall, F1, specificity, sensitivity, AUC, ROC curves, PR curves, confusion matrix, classification report -- for any model in `MODEL_REGISTRY`.

In [ ]:
"""
eval_pipeline.py — Stage 3
============================
Generic `evaluate_model(model_name, cfg, test_loader)` that runs on ANY of
the 7 architectures in MODEL_REGISTRY and produces every metric your spec
asks for:

    Accuracy, Precision, Recall, F1 (macro + weighted)
    Specificity, Sensitivity (per class, derived from the confusion matrix)
    AUC score (multi-class, one-vs-rest, macro + per-class)
    ROC curve (per class + macro-average) — saved as PNG
    Precision-Recall curve (per class) — saved as PNG
    Confusion matrix — saved as PNG
    Full classification report — saved as TXT

Design notes
------------
This generalizes your original `full_evaluation()` cell (which already
computed accuracy / weighted-F1 / macro-F1 / per-class accuracy /
confusion-matrix / classification-report for ONE model). Nothing from
that logic is thrown away — it's the base this extends. What's new here:

  1. Parameterized by `model_name` so it works identically for all 7
     models in MODEL_REGISTRY (uses `build_model()` + `load_ckpt()` from
     your existing models.py / train_pipeline.py — Stage 1 & 2 code —
     unchanged).
  2. Adds precision, recall, specificity, sensitivity, AUC, ROC, and PR
     curves, which your original cell did not compute.
  3. Every model's outputs land in their own folder under
     cfg.EVAL_DIR/<model_name>/, so results never overwrite each other —
     required for Stage 6's comparison tables.
  4. Returns a flat dict of scalar metrics (accuracy, precision, recall,
     f1, specificity, sensitivity, auc, ...) that Stage 6 can directly
     assemble into the CSV/Excel comparison table without recomputation.

Paste this as its own cell AFTER the models.py cell (Stage 1) and the
train_pipeline.py cell (Stage 2). Requires `build_model`, `load_ckpt`,
`cfg`, `MODEL_REGISTRY` to already exist in the kernel — all of them do,
from your earlier cells.
"""

import os
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve,
)
from sklearn.preprocessing import label_binarize


# ─────────────────────────────────────────────────────────────────────────
# Core: run inference once, collect labels / preds / probabilities
# ─────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def _run_inference(model, loader, cfg):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []

    for imgs, labels in loader:
        imgs = imgs.to(cfg.DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits, _ = model(imgs)
        probs = F.softmax(logits.float(), dim=-1)

        all_probs.append(probs.cpu().numpy())
        all_preds.extend(probs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

    probs = np.concatenate(all_probs, axis=0)
    preds = np.array(all_preds)
    labels = np.array(all_labels)
    return probs, preds, labels


# ─────────────────────────────────────────────────────────────────────────
# Specificity / Sensitivity per class from the confusion matrix
# ─────────────────────────────────────────────────────────────────────────
def _specificity_sensitivity(cm):
    """
    For each class c (one-vs-rest on the multi-class confusion matrix):
        TP = cm[c, c]
        FN = sum(cm[c, :]) - TP
        FP = sum(cm[:, c]) - TP
        TN = total - TP - FN - FP

        sensitivity (recall)   = TP / (TP + FN)
        specificity            = TN / (TN + FP)
    """
    n = cm.shape[0]
    total = cm.sum()
    sens, spec = np.zeros(n), np.zeros(n)
    for c in range(n):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = total - tp - fn - fp
        sens[c] = tp / (tp + fn + 1e-9)
        spec[c] = tn / (tn + fp + 1e-9)
    return sens, spec


# ─────────────────────────────────────────────────────────────────────────
# Plot helpers — each returns the saved path
# ─────────────────────────────────────────────────────────────────────────
def _plot_confusion_matrix(cm, class_names, out_path, model_name):
    short = [n.replace("ily", "").replace("fully", "").replace("rily", "")
             for n in class_names]
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=short, yticklabels=short,
                linewidths=0.5, ax=ax)
    ax.set_title(f"Confusion Matrix — {model_name}",
                 fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    return out_path


def _plot_roc_curves(labels, probs, class_names, out_path, model_name):
    n_classes = len(class_names)
    y_bin = label_binarize(labels, classes=list(range(n_classes)))

    fpr, tpr, roc_auc_per_class = {}, {}, {}
    for c in range(n_classes):
        fpr[c], tpr[c], _ = roc_curve(y_bin[:, c], probs[:, c])
        roc_auc_per_class[c] = auc(fpr[c], tpr[c])

    # macro-average ROC
    all_fpr = np.unique(np.concatenate([fpr[c] for c in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)

    fig, ax = plt.subplots(figsize=(9, 8))
    for c in range(n_classes):
        ax.plot(fpr[c], tpr[c], lw=1, alpha=0.6,
                label=f"{class_names[c]} (AUC={roc_auc_per_class[c]:.3f})")
    ax.plot(all_fpr, mean_tpr, color="black", lw=2.5, linestyle="--",
            label=f"Macro-average (AUC={macro_auc:.3f})")
    ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle=":")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves — {model_name}", fontweight="bold")
    ax.legend(fontsize=7, loc="lower right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()

    return out_path, roc_auc_per_class, macro_auc


def _plot_pr_curves(labels, probs, class_names, out_path, model_name):
    n_classes = len(class_names)
    y_bin = label_binarize(labels, classes=list(range(n_classes)))

    fig, ax = plt.subplots(figsize=(9, 8))
    for c in range(n_classes):
        prec, rec, _ = precision_recall_curve(y_bin[:, c], probs[:, c])
        ax.plot(rec, prec, lw=1, alpha=0.7, label=class_names[c])
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"Precision-Recall Curves — {model_name}", fontweight="bold")
    ax.legend(fontsize=7, loc="lower left")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    return out_path


# ─────────────────────────────────────────────────────────────────────────
# Main entry point — evaluate ONE model
# ─────────────────────────────────────────────────────────────────────────
def evaluate_model(model_name, cfg, test_loader,
                    ckpt_name=None, class_names=None):
    """
    Loads {model_name}_stage3_best.pth (or a specified ckpt_name),
    runs it on test_loader, computes every metric, saves every plot
    into cfg.EVAL_DIR/<model_name>/, and returns a flat metrics dict.
    """
    ckpt_name = ckpt_name or f"{model_name}_stage3_best.pth"
    class_names = class_names or cfg.COMPOUND_CLASSES

    out_dir = os.path.join(cfg.EVAL_DIR, model_name)
    os.makedirs(out_dir, exist_ok=True)

    model = build_model(model_name, cfg)
    load_ckpt(model, ckpt_name)

    probs, preds, labels = _run_inference(model, test_loader, cfg)
    n_classes = len(class_names)

    # ── Scalar metrics ────────────────────────────────────────────────
    acc        = accuracy_score(labels, preds) * 100
    prec_macro = precision_score(labels, preds, average="macro", zero_division=0) * 100
    prec_w     = precision_score(labels, preds, average="weighted", zero_division=0) * 100
    rec_macro  = recall_score(labels, preds, average="macro", zero_division=0) * 100
    rec_w      = recall_score(labels, preds, average="weighted", zero_division=0) * 100
    f1_macro   = f1_score(labels, preds, average="macro", zero_division=0) * 100
    f1_w       = f1_score(labels, preds, average="weighted", zero_division=0) * 100

    cm = confusion_matrix(labels, preds, labels=list(range(n_classes)))
    sens, spec = _specificity_sensitivity(cm)

    try:
        y_bin = label_binarize(labels, classes=list(range(n_classes)))
        auc_macro = roc_auc_score(y_bin, probs, average="macro", multi_class="ovr") * 100
    except ValueError:
        auc_macro = float("nan")  # happens if a class is entirely absent from the split

    report = classification_report(
        labels, preds, target_names=class_names,
        labels=list(range(n_classes)), digits=4, zero_division=0,
    )

    # ── Plots ──────────────────────────────────────────────────────────
    cm_path = _plot_confusion_matrix(
        cm, class_names, os.path.join(out_dir, "confusion_matrix.png"), model_name)
    roc_path, auc_per_class, roc_macro_auc = _plot_roc_curves(
        labels, probs, class_names, os.path.join(out_dir, "roc_curve.png"), model_name)
    pr_path = _plot_pr_curves(
        labels, probs, class_names, os.path.join(out_dir, "pr_curve.png"), model_name)

    with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
        f.write(report)

    # ── Assemble result dict (this is what Stage 6 tables will consume) ─
    metrics = {
        "model": model_name,
        "accuracy": round(acc, 2),
        "precision_macro": round(prec_macro, 2),
        "precision_weighted": round(prec_w, 2),
        "recall_macro": round(rec_macro, 2),
        "recall_weighted": round(rec_w, 2),
        "f1_macro": round(f1_macro, 2),
        "f1_weighted": round(f1_w, 2),
        "sensitivity_macro": round(float(sens.mean()) * 100, 2),
        "specificity_macro": round(float(spec.mean()) * 100, 2),
        "auc_macro": round(auc_macro, 2),
        "per_class": {
            class_names[c]: {
                "sensitivity": round(float(sens[c]) * 100, 2),
                "specificity": round(float(spec[c]) * 100, 2),
                "auc": round(float(auc_per_class[c]) * 100, 2),
            }
            for c in range(n_classes)
        },
        "confusion_matrix": cm.tolist(),
        "paths": {
            "confusion_matrix": cm_path,
            "roc_curve": roc_path,
            "pr_curve": pr_path,
            "classification_report": os.path.join(out_dir, "classification_report.txt"),
        },
    }

    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"\n{'='*65}")
    print(f"  {model_name} — EVALUATION COMPLETE")
    print(f"{'='*65}")
    print(f"  Accuracy            : {acc:.2f}%")
    print(f"  Precision (macro)   : {prec_macro:.2f}%")
    print(f"  Recall (macro)      : {rec_macro:.2f}%")
    print(f"  F1 (macro)          : {f1_macro:.2f}%")
    print(f"  Sensitivity (macro) : {metrics['sensitivity_macro']:.2f}%")
    print(f"  Specificity (macro) : {metrics['specificity_macro']:.2f}%")
    print(f"  AUC (macro)         : {auc_macro:.2f}%")
    print(f"  Outputs → {out_dir}")

    return metrics


# ─────────────────────────────────────────────────────────────────────────
# Evaluate every model in a list, save a combined summary
# ─────────────────────────────────────────────────────────────────────────
def evaluate_all_models(MODEL_LIST, cfg, test_loader, class_names=None):
    all_metrics = {}
    summary_path = os.path.join(cfg.EVAL_DIR, "all_models_eval_summary.json")

    for name in MODEL_LIST:
        all_metrics[name] = evaluate_model(name, cfg, test_loader, class_names=class_names)
        with open(summary_path, "w") as f:
            json.dump(all_metrics, f, indent=2)

    print(f"\n\n{'='*65}")
    print(f"  ALL MODELS EVALUATED")
    print(f"{'='*65}")
    for name, m in all_metrics.items():
        print(f"  {name:20s} acc={m['accuracy']:.2f}%  f1={m['f1_macro']:.2f}%  "
              f"auc={m['auc_macro']:.2f}%")
    print(f"\n  Summary → {summary_path}")
    return all_metrics


# ─────────────────────────────────────────────────────────────────────────
# Example usage (uncomment once your test_loader is ready):
#
# MODEL_LIST = list(MODEL_REGISTRY.keys())   # all 7, or a subset
# eval_summary = evaluate_all_models(MODEL_LIST, cfg, test_loader)
# ─────────────────────────────────────────────────────────────────────────


## Stage 4 -- Training / Validation Curves

In [ ]:
"""
plotting_pipeline.py — Stage 4
================================
Generates the training/validation accuracy and loss curves your TOC asks
for, for any model in MODEL_REGISTRY, reading the per-model training log
that Stage 2's `train_model()` already writes to
cfg.LOG_DIR/<model_name>_training_log.json.

REQUIRED PATCH to run_stage() (Stage 2 cell, reused-unchanged utility)
-----------------------------------------------------------------------
Your original `run_stage()` computes `tr_loss` and `vl_loss` every epoch
but never stores them in the log — only tr_acc / vl_acc / vl_f1 were
saved. Loss curves are explicitly in your TOC, so this one line has to
change. Everything else in run_stage is untouched.

Find this line inside run_stage():

    log.append({"epoch": epoch, "stage": stage,
                "tr_acc": tr_acc, "vl_acc": vl_acc, "vl_f1": vl_f1})

Replace it with:

    log.append({"epoch": epoch, "stage": stage,
                "tr_loss": tr_loss, "vl_loss": vl_loss,
                "tr_acc": tr_acc, "vl_acc": vl_acc, "vl_f1": vl_f1,
                "elapsed_sec": elapsed})

(`elapsed` is already computed a few lines above via `time.time() - t0`.)
The `elapsed_sec` field is what Stage 5's benchmarking uses to report
total training time per model — without it we'd have no record of how
long each model actually took to train.

If you've already trained models with the OLD log format, this file's
plotting functions still work — they just skip the loss curve for any
log that doesn't have "tr_loss" and print a warning instead of crashing.
"""

import os
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def plot_training_curves(model_name, cfg):
    """
    Reads cfg.LOG_DIR/<model_name>_training_log.json and saves:
        - <model_name>_accuracy_curve.png  (train + val accuracy)
        - <model_name>_loss_curve.png      (train + val loss, if logged)
    into cfg.EVAL_DIR/<model_name>/.
    Returns dict of saved paths.
    """
    log_path = os.path.join(cfg.LOG_DIR, f"{model_name}_training_log.json")
    if not os.path.exists(log_path):
        print(f"  ⚠  No training log found for {model_name} at {log_path}")
        return {}

    with open(log_path) as f:
        log = json.load(f)

    out_dir = os.path.join(cfg.EVAL_DIR, model_name)
    os.makedirs(out_dir, exist_ok=True)

    # cumulative epoch index across stages (stage 1 -> 2 -> 3), for a
    # single continuous x-axis
    cum_epoch = list(range(1, len(log) + 1))
    stage_boundaries = []
    prev_stage = None
    for i, e in enumerate(log):
        if e["stage"] != prev_stage:
            stage_boundaries.append(i)
            prev_stage = e["stage"]

    tr_acc = [e["tr_acc"] for e in log]
    vl_acc = [e["vl_acc"] for e in log]

    paths = {}

    # ── Accuracy curve ──────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(cum_epoch, tr_acc, color="#2E86AB", linewidth=1.8, label="Train Accuracy")
    ax.plot(cum_epoch, vl_acc, color="#E84855", linewidth=1.8, label="Val Accuracy")
    for b in stage_boundaries[1:]:
        ax.axvline(cum_epoch[b], color="gray", linestyle=":", linewidth=1)
    ax.set_xlabel("Epoch (cumulative across stages)")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"Training / Validation Accuracy — {model_name}", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    acc_path = os.path.join(out_dir, "accuracy_curve.png")
    plt.savefig(acc_path, dpi=150, bbox_inches="tight")
    plt.close()
    paths["accuracy_curve"] = acc_path

    # ── Loss curve (only if the log has it — see patch note above) ──
    if "tr_loss" in log[0] and "vl_loss" in log[0]:
        tr_loss = [e["tr_loss"] for e in log]
        vl_loss = [e["vl_loss"] for e in log]

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(cum_epoch, tr_loss, color="#2E86AB", linewidth=1.8, label="Train Loss")
        ax.plot(cum_epoch, vl_loss, color="#E84855", linewidth=1.8, label="Val Loss")
        for b in stage_boundaries[1:]:
            ax.axvline(cum_epoch[b], color="gray", linestyle=":", linewidth=1)
        ax.set_xlabel("Epoch (cumulative across stages)")
        ax.set_ylabel("Loss")
        ax.set_title(f"Training / Validation Loss — {model_name}", fontweight="bold")
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        loss_path = os.path.join(out_dir, "loss_curve.png")
        plt.savefig(loss_path, dpi=150, bbox_inches="tight")
        plt.close()
        paths["loss_curve"] = loss_path
    else:
        print(f"  ⚠  {model_name}: log has no tr_loss/vl_loss — apply the "
              f"run_stage patch above and retrain to get the loss curve.")

    print(f"  {model_name}: plots saved → {out_dir}")
    return paths


def plot_all_training_curves(MODEL_LIST, cfg):
    all_paths = {}
    for name in MODEL_LIST:
        all_paths[name] = plot_training_curves(name, cfg)
    return all_paths


# ─────────────────────────────────────────────────────────────────────────
# Example usage:
#
# MODEL_LIST = list(MODEL_REGISTRY.keys())
# curve_paths = plot_all_training_curves(MODEL_LIST, cfg)
# ─────────────────────────────────────────────────────────────────────────


## Stage 5 -- Benchmarking (Params, FLOPs, MACs, Size, Speed)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "thop", "--break-system-packages"])


In [ ]:
"""
benchmarking_pipeline.py — Stage 5
=====================================
Generic `benchmark_model(model_name, cfg)` that measures, for any of the
7 architectures in MODEL_REGISTRY:

    Total Parameters
    Trainable Parameters (at full unfreeze, i.e. stage3_setup())
    FLOPs / MACs               (via thop)
    Model Size (MB)            (serialized state_dict on disk)
    Training Time (sec)        (summed from the per-epoch "elapsed_sec"
                                 field — requires the Stage 4 run_stage
                                 patch; falls back to "unknown" otherwise)
    Inference Time per Image (ms)
    FPS (Frames Per Second)

Requires: pip install thop --break-system-packages
"""

import os
import json
import time
import torch

try:
    from thop import profile as _thop_profile
except ImportError:
    _thop_profile = None
    print("  ⚠  thop not installed — run: "
          "pip install thop --break-system-packages")


def _get_training_time(model_name, cfg):
    log_path = os.path.join(cfg.LOG_DIR, f"{model_name}_training_log.json")
    if not os.path.exists(log_path):
        return None
    with open(log_path) as f:
        log = json.load(f)
    if not log or "elapsed_sec" not in log[0]:
        return None
    return float(sum(e["elapsed_sec"] for e in log))


def _model_size_mb(model):
    tmp_path = "/tmp/_benchmark_tmp.pth"
    torch.save(model.state_dict(), tmp_path)
    size_mb = os.path.getsize(tmp_path) / 1e6
    os.remove(tmp_path)
    return size_mb


@torch.no_grad()
def _measure_inference_time(model, cfg, n_warmup=10, n_runs=100):
    model.eval()
    dummy = torch.randn(1, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE, device=cfg.DEVICE)

    for _ in range(n_warmup):
        _ = model(dummy)
    if cfg.DEVICE.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(n_runs):
        _ = model(dummy)
    if cfg.DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0

    per_image_ms = (elapsed / n_runs) * 1000
    fps = n_runs / elapsed
    return per_image_ms, fps


def benchmark_model(model_name, cfg, ckpt_name=None, n_warmup=10, n_runs=100):
    """
    Returns a flat dict of benchmark metrics for one model. Does NOT
    require a trained checkpoint for params/FLOPs/inference-time — those
    depend only on architecture. Training time is read from the log if
    available; if a checkpoint exists it's loaded so inference-time
    benchmarking reflects the actual deployed weights (numerically
    irrelevant for timing, but keeps behavior consistent with eval).
    """
    model = build_model(model_name, cfg)
    model.stage3_setup()  # fully unfrozen — "trainable params" == full capacity used in training

    ckpt_name = ckpt_name or f"{model_name}_stage3_best.pth"
    ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, ckpt_name)
    if os.path.exists(ckpt_path):
        load_ckpt(model, ckpt_name)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    flops, macs = None, None
    if _thop_profile is not None:
        dummy = torch.randn(1, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE, device=cfg.DEVICE)
        try:
            macs_raw, params_raw = _thop_profile(model, inputs=(dummy,), verbose=False)
            macs = macs_raw
            flops = macs_raw * 2  # standard convention: 1 MAC == 2 FLOPs
        except Exception as e:
            print(f"  ⚠  {model_name}: thop profiling failed ({e}) — "
                  f"FLOPs/MACs left as None")

    model_size_mb = _model_size_mb(model)
    infer_ms, fps = _measure_inference_time(model, cfg, n_warmup, n_runs)
    train_time_sec = _get_training_time(model_name, cfg)

    metrics = {
        "model": model_name,
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "macs": int(macs) if macs is not None else None,
        "flops": int(flops) if flops is not None else None,
        "model_size_mb": round(model_size_mb, 2),
        "training_time_sec": round(train_time_sec, 1) if train_time_sec is not None else None,
        "inference_time_ms": round(infer_ms, 3),
        "fps": round(fps, 1),
    }

    print(f"\n{'='*65}")
    print(f"  {model_name} — BENCHMARK")
    print(f"{'='*65}")
    print(f"  Total params       : {total_params:,}")
    print(f"  Trainable params    : {trainable_params:,}")
    if flops is not None:
        print(f"  FLOPs               : {flops/1e9:.2f} GFLOPs")
        print(f"  MACs                : {macs/1e9:.2f} GMACs")
    print(f"  Model size          : {model_size_mb:.2f} MB")
    print(f"  Training time       : "
          f"{train_time_sec:.1f}s" if train_time_sec is not None else "  Training time       : unknown (retrain with Stage 4 run_stage patch)")
    print(f"  Inference time/img  : {infer_ms:.3f} ms")
    print(f"  FPS                 : {fps:.1f}")

    out_dir = os.path.join(cfg.EVAL_DIR, model_name)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "benchmark.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    return metrics


def benchmark_all_models(MODEL_LIST, cfg, n_warmup=10, n_runs=100):
    all_bench = {}
    summary_path = os.path.join(cfg.EVAL_DIR, "all_models_benchmark_summary.json")
    for name in MODEL_LIST:
        all_bench[name] = benchmark_model(name, cfg, n_warmup=n_warmup, n_runs=n_runs)
        with open(summary_path, "w") as f:
            json.dump(all_bench, f, indent=2)
    print(f"\n  Benchmark summary → {summary_path}")
    return all_bench


# ─────────────────────────────────────────────────────────────────────────
# Example usage:
#
# MODEL_LIST = list(MODEL_REGISTRY.keys())
# bench_summary = benchmark_all_models(MODEL_LIST, cfg)
# ─────────────────────────────────────────────────────────────────────────


## Stage 6 -- Comparison Tables (CSV + Excel)

In [ ]:
"""
comparison_tables.py — Stage 6
=================================
Builds the two comparison tables your spec calls for, from the outputs
of Stage 3 (eval_summary) and Stage 5 (bench_summary), and saves both as
CSV and as a single multi-sheet Excel workbook.

Table 1 — Performance comparison:
    Model | Accuracy | Precision | Recall | F1 | AUC

Table 2 — Efficiency comparison:
    Model | Params | Trainable Params | FLOPs | MACs | Model Size |
    Train Time | Inference Time | FPS

Requires: pip install openpyxl --break-system-packages  (usually already present)
"""

import os
import pandas as pd


def build_performance_table(eval_summary):
    rows = []
    for name, m in eval_summary.items():
        rows.append({
            "Model": name,
            "Accuracy (%)": m["accuracy"],
            "Precision (macro %)": m["precision_macro"],
            "Recall (macro %)": m["recall_macro"],
            "F1 (macro %)": m["f1_macro"],
            "Sensitivity (macro %)": m["sensitivity_macro"],
            "Specificity (macro %)": m["specificity_macro"],
            "AUC (macro %)": m["auc_macro"],
        })
    return pd.DataFrame(rows).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)


def build_efficiency_table(bench_summary):
    rows = []
    for name, b in bench_summary.items():
        rows.append({
            "Model": name,
            "Total Params": b["total_params"],
            "Trainable Params": b["trainable_params"],
            "FLOPs (G)": round(b["flops"] / 1e9, 3) if b["flops"] is not None else None,
            "MACs (G)": round(b["macs"] / 1e9, 3) if b["macs"] is not None else None,
            "Model Size (MB)": b["model_size_mb"],
            "Training Time (s)": b["training_time_sec"],
            "Inference Time (ms/img)": b["inference_time_ms"],
            "FPS": b["fps"],
        })
    return pd.DataFrame(rows).sort_values("Total Params").reset_index(drop=True)


def generate_comparison_tables(eval_summary, bench_summary, cfg):
    perf_df = build_performance_table(eval_summary)
    eff_df = build_efficiency_table(bench_summary)

    out_dir = cfg.EVAL_DIR
    os.makedirs(out_dir, exist_ok=True)

    perf_csv = os.path.join(out_dir, "performance_comparison.csv")
    eff_csv = os.path.join(out_dir, "efficiency_comparison.csv")
    perf_df.to_csv(perf_csv, index=False)
    eff_df.to_csv(eff_csv, index=False)

    xlsx_path = os.path.join(out_dir, "model_comparison.xlsx")
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        perf_df.to_excel(writer, sheet_name="Performance", index=False)
        eff_df.to_excel(writer, sheet_name="Efficiency", index=False)

    print(f"  Performance table → {perf_csv}")
    print(f"  Efficiency table  → {eff_csv}")
    print(f"  Combined workbook → {xlsx_path}")

    print("\n" + "="*80)
    print("PERFORMANCE COMPARISON")
    print("="*80)
    print(perf_df.to_string(index=False))
    print("\n" + "="*80)
    print("EFFICIENCY COMPARISON")
    print("="*80)
    print(eff_df.to_string(index=False))

    return perf_df, eff_df


# ─────────────────────────────────────────────────────────────────────────
# Example usage (after Stage 3's evaluate_all_models and
# Stage 5's benchmark_all_models have both run):
#
# perf_df, eff_df = generate_comparison_tables(eval_summary, bench_summary, cfg)
# ─────────────────────────────────────────────────────────────────────────


## Stage 7 -- Ablation Study

In [ ]:
"""
ablation_study.py — Stage 7
==============================
Your spec asks for 4 ablations of the Proposed model:

    Without Morphological Splicing
    Without SA-CNN
    Without ALSTM
    Without Vision Transformer

Component ablations (free — already exist in MODEL_REGISTRY)
----------------------------------------------------------------
The Proposed model is the 3-branch fusion SA-CNN + ALSTM + ViT. Removing
one branch at a time gives you exactly the 2-branch models already built
in Stage 1 — no new architecture code needed:

    Without SA-CNN  →  MODEL_REGISTRY["ALSTM_ViT"]
    Without ALSTM   →  MODEL_REGISTRY["SACNN_ViT"]
    Without ViT     →  MODEL_REGISTRY["SACNN_ALSTM"]

So three of the four ablations are just: train + evaluate those three
models (which Stage 2 / Stage 3 already do) and compare their metrics
against "Proposed" in the Stage 6 table. This file's `ablation_summary()`
does exactly that comparison automatically once all four have been
trained and evaluated.

Data ablation (needs YOUR splice code)
----------------------------------------
"Without Morphological Splicing" is NOT a model change — it means
training the SAME Proposed architecture on a training set that skips
the splice step (e.g. plain single-emotion images, or a different
compound-construction method). This can't be built without your
`SyntheticCompoundDataset` / `get_dataloaders()` code, which — same as
flagged earlier — isn't present in either notebook you've shared.

`run_ablation_study()` below is written generically: you pass it a dict
of {ablation_name: (model_name, train_loader, val_loader)}. For the 3
component ablations, reuse your normal spliced train_loader/val_loader.
For "Without Splicing", pass a train_loader built from your own
non-spliced dataset variant — plug it in and the rest of the pipeline
(training, evaluation, comparison) runs unchanged.
"""

import os
import json


def run_ablation_study(ablation_configs, cfg, test_loader,
                        stage_epochs=(25, 25, 10)):
    """
    ablation_configs: dict like
        {
          "Without SA-CNN":  ("ALSTM_ViT",   train_loader, val_loader),
          "Without ALSTM":   ("SACNN_ViT",   train_loader, val_loader),
          "Without ViT":     ("SACNN_ALSTM", train_loader, val_loader),
          "Without Splicing": ("Proposed", train_loader_no_splice, val_loader_no_splice),
        }
    Trains + evaluates each config, using the SAME `train_model` (Stage 2)
    and `evaluate_model` (Stage 3) functions as the main comparison — so
    the ablation results are directly comparable to the main table.

    Returns: {ablation_name: {"train_metrics":..., "eval_metrics":...}}
    """
    results = {}
    out_path = os.path.join(cfg.EVAL_DIR, "ablation_study_results.json")

    for ablation_name, (model_name, train_loader, val_loader) in ablation_configs.items():
        print(f"\n{'#'*70}")
        print(f"#  ABLATION: {ablation_name}  (architecture: {model_name})")
        print(f"{'#'*70}")

        ckpt_tag = f"ablation_{ablation_name.replace(' ', '_')}"
        best_acc, ckpt_path, _ = train_model(
            model_name, train_loader, val_loader, cfg, stage_epochs=stage_epochs
        )
        # evaluate_model expects a checkpoint name matching model_name's
        # default pattern; since ablations reuse existing architectures,
        # this evaluates the checkpoint just produced above.
        eval_metrics = evaluate_model(model_name, cfg, test_loader)

        results[ablation_name] = {
            "architecture": model_name,
            "best_val_acc": best_acc,
            "eval_metrics": eval_metrics,
        }
        with open(out_path, "w") as f:
            json.dump(results, f, indent=2)

    return results


def ablation_summary(ablation_results, proposed_eval_metrics):
    """
    Prints and returns a table comparing each ablation's accuracy/F1
    against the full Proposed model, with the performance drop (Δ) —
    the core "ablation study" comparison your TOC asks for.
    """
    import pandas as pd

    rows = [{
        "Configuration": "Proposed (Full Model)",
        "Accuracy (%)": proposed_eval_metrics["accuracy"],
        "F1 macro (%)": proposed_eval_metrics["f1_macro"],
        "Δ Accuracy": 0.0,
        "Δ F1": 0.0,
    }]
    for name, r in ablation_results.items():
        m = r["eval_metrics"]
        rows.append({
            "Configuration": name,
            "Accuracy (%)": m["accuracy"],
            "F1 macro (%)": m["f1_macro"],
            "Δ Accuracy": round(m["accuracy"] - proposed_eval_metrics["accuracy"], 2),
            "Δ F1": round(m["f1_macro"] - proposed_eval_metrics["f1_macro"], 2),
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


# ─────────────────────────────────────────────────────────────────────────
# Example usage (after main training/eval for "Proposed" is done):
#
# ablation_configs = {
#     "Without SA-CNN":  ("ALSTM_ViT",   train_loader, val_loader),
#     "Without ALSTM":   ("SACNN_ViT",   train_loader, val_loader),
#     "Without ViT":     ("SACNN_ALSTM", train_loader, val_loader),
#     # "Without Splicing": ("Proposed", train_loader_no_splice, val_loader_no_splice),
# }
# ablation_results = run_ablation_study(ablation_configs, cfg, test_loader)
# summary_df = ablation_summary(ablation_results, eval_summary["Proposed"])
# ─────────────────────────────────────────────────────────────────────────


## Dataset-wise Performance Analysis (JAFFE, CK+, RAF-DB, AffectNet, FER-2013)

Generalized from your original `run_all_dataset_evaluations_v2()` cell to work
with any model in `MODEL_REGISTRY`. FER-2013 is skipped gracefully until you
paste in `FER2013Pool` (see the module docstring below).


In [ ]:
"""
dataset_wise_evaluation.py
=============================
Generalizes your existing `run_all_dataset_evaluations_v2()` cell (which
evaluated ONE hardcoded `model` variable) so it works for any model_name
in MODEL_REGISTRY — needed for the "Dataset-wise Performance Analysis
(JAFFE, CK+, RAF-DB, AffectNet, FER-2013)" section in your TOC.

Everything below — FOLDER_TO_BASIC_EXT, collect_pool_ext,
SyntheticEvalDataset, evaluate_dataset, build_all_pools_v2 — is your
original v2 logic, copied verbatim except:
  1. evaluate_dataset() now saves its confusion matrix into
     cfg.EVAL_DIR/<model_name>/ instead of directly into cfg.EVAL_DIR,
     so results from different models don't overwrite each other.
  2. run_all_dataset_evaluations() takes `model_name` and builds +
     loads that model via build_model()/load_ckpt(), instead of
     assuming a single global `model`.

⚠ MISSING DEPENDENCY — FER2013Pool
------------------------------------
`build_all_pools_v2()` calls `FER2013Pool(split="train")`, which is
referenced in both notebooks you've shared but never defined in either.
Until you paste in that class, `build_all_pools_v2()` will raise
NameError on the FER-2013 pool specifically — everything else (JAFFE,
CK+, RAF-DB, AffectNet) will still work. A `try/except` below skips
FER-2013 gracefully rather than crashing the whole run; remove the
try/except once you've added FER2013Pool.
"""

import os
import random
import json
import numpy as np
import torch
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns


FOLDER_TO_BASIC_EXT = {
    "anger": 0, "angry": 0, "an": 0,
    "disgust": 1, "disgusted": 1, "di": 1,
    "fear": 2, "fearful": 2, "fe": 2,
    "happy": 3, "happiness": 3, "ha": 3,
    "sad": 4, "sadness": 4, "sa": 4,
    "surprise": 5, "surprised": 5, "su": 5,
    "neutral": 6, "ne": 6,
    "0": 6, "1": 3, "2": 4, "3": 5, "4": 2, "5": 1, "6": 0,
}
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".pgm")


def collect_pool_ext(root_dir, cfg, max_per_class=999):
    by_class = {i: [] for i in range(7)}
    if not os.path.exists(root_dir):
        print(f"  ⚠  Not found: {root_dir}")
        return by_class
    for folder in sorted(os.listdir(root_dir)):
        idx = FOLDER_TO_BASIC_EXT.get(folder.lower().strip())
        if idx is None:
            continue
        class_dir = os.path.join(root_dir, folder)
        if not os.path.isdir(class_dir):
            continue
        paths = [os.path.join(class_dir, f)
                 for f in os.listdir(class_dir)
                 if f.lower().endswith(IMG_EXTS)]
        random.shuffle(paths)
        by_class[idx].extend(paths[:max_per_class])
        print(f"    '{folder}' → class {idx} "
              f"({cfg.BASIC_CLASSES[idx]}): {len(by_class[idx])} imgs")
    total = sum(len(v) for v in by_class.values())
    print(f"  Total: {total} images")
    return by_class


class SyntheticEvalDataset(Dataset):
    def __init__(self, by_class, cfg, samples_per_class=100, transform=None):
        self.cfg = cfg
        self.transform = transform
        self.samples = self._generate(by_class, samples_per_class)

    def _generate(self, by_class, n):
        samples = []
        for compound_idx, (e1, e2) in self.cfg.COMPOUND_TO_BASIC.items():
            pool1 = by_class.get(e1, [])
            pool2 = by_class.get(e2, [])
            if len(pool1) < 2 or len(pool2) < 2:
                continue
            actual_n = min(n, len(pool1), len(pool2))
            for _ in range(actual_n):
                samples.append((random.choice(pool1), random.choice(pool2), compound_idx))
        return samples

    def _load(self, path):
        img = cv2.imread(path)
        if img is None:
            img = np.array(Image.open(path).convert("RGB"))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return cv2.resize(img, (self.cfg.IMAGE_SIZE, self.cfg.IMAGE_SIZE))

    def _splice(self, img1, img2):
        h = img1.shape[0]
        boundary = int(h * random.uniform(0.45, 0.55))
        decay = 10
        result = img1.copy().astype(np.float32)
        result[boundary:] = img2[boundary:].astype(np.float32)
        for d in range(decay):
            row = boundary - decay // 2 + d
            if 0 <= row < h:
                t = d / decay
                result[row] = ((1 - t) * img1[row].astype(np.float32)
                                + t * img2[row].astype(np.float32))
        return np.clip(result, 0, 255).astype(np.uint8)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p1, p2, label = self.samples[idx]
        mixed = self._splice(self._load(p1), self._load(p2))
        if self.transform:
            mixed = self.transform(image=mixed)["image"]
        return mixed, torch.tensor(label, dtype=torch.long)


def evaluate_dataset(model, by_class, dataset_name, cfg, model_name,
                      samples_per_class=100, batch_size=32):
    print(f"\n{'─'*60}\n  Evaluating: {dataset_name}\n{'─'*60}")
    ds = SyntheticEvalDataset(by_class, cfg, samples_per_class, transform=get_val_transforms())
    if len(ds) == 0:
        print("  ✗ No samples — skipping.")
        return None

    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits, _ = model(imgs.to(cfg.DEVICE))
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    preds, labels = np.array(all_preds), np.array(all_labels)
    present = sorted(set(labels.tolist()))
    p_names = [cfg.COMPOUND_CLASSES[i] for i in present]

    acc = accuracy_score(labels, preds) * 100
    f1_w = f1_score(labels, preds, average="weighted", zero_division=0) * 100
    f1_m = f1_score(labels, preds, average="macro", zero_division=0) * 100

    cm = confusion_matrix(labels, preds, labels=present)
    pca = cm.diagonal() / (cm.sum(axis=1) + 1e-9) * 100

    out_dir = os.path.join(cfg.EVAL_DIR, model_name)
    os.makedirs(out_dir, exist_ok=True)
    short = [n.replace("ily", "").replace("fully", "").replace("rily", "") for n in p_names]
    sz = max(8, len(present))
    fig, ax = plt.subplots(figsize=(sz, sz - 1))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=short,
                yticklabels=short, linewidths=0.5, ax=ax)
    ax.set_title(f"Confusion Matrix — {model_name} on {dataset_name}", fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    safe = dataset_name.replace(" ", "_").replace("/", "-").replace("+", "plus")
    cm_path = os.path.join(out_dir, f"cm_{safe}.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"  Accuracy={acc:.2f}%  W-F1={f1_w:.2f}%  M-F1={f1_m:.2f}%  N={len(preds)}")

    return {
        "dataset": dataset_name, "n_samples": int(len(preds)),
        "accuracy": round(acc, 2), "weighted_f1": round(f1_w, 2), "macro_f1": round(f1_m, 2),
        "per_class_accuracy": {n: round(float(a), 2) for n, a in zip(p_names, pca)},
    }


def build_all_pools_v2(cfg):
    pools = {}
    pools["JAFFE"] = collect_pool_ext(cfg.JAFFE_DIR, cfg, max_per_class=999)
    pools["CK+"] = collect_pool_ext(cfg.CKP_DIR, cfg, max_per_class=999)
    pools["RAF-DB (train)"] = collect_pool_ext(os.path.join(cfg.RAF_DB_DIR, "train"), cfg, max_per_class=500)
    pools["RAF-DB (test)"] = collect_pool_ext(os.path.join(cfg.RAF_DB_DIR, "test"), cfg, max_per_class=500)
    pools["AffectNet"] = collect_pool_ext(os.path.join(cfg.AFFECTNET_DIR, "Test"), cfg, max_per_class=500)

    try:
        fer_pool = FER2013Pool(split="train")  # noqa: F821 — see module docstring
        pools["FER-2013"] = fer_pool.by_class
    except NameError:
        print("  ⚠  FER2013Pool not defined — skipping FER-2013 "
              "(paste your original FER2013Pool class to include it)")
    return pools


def run_dataset_wise_evaluation(model_name, cfg, ckpt_name=None, samples_per_class=100):
    ckpt_name = ckpt_name or f"{model_name}_stage3_best.pth"
    model = build_model(model_name, cfg)
    load_ckpt(model, ckpt_name)
    model.eval()

    pools = build_all_pools_v2(cfg)
    all_res = []
    for ds_name, by_class in pools.items():
        if sum(len(v) for v in by_class.values()) == 0:
            print(f"  ⚠  {ds_name}: pool empty — skipping.")
            continue
        res = evaluate_dataset(model, by_class, ds_name, cfg, model_name,
                                samples_per_class=samples_per_class)
        if res:
            all_res.append(res)

    out_dir = os.path.join(cfg.EVAL_DIR, model_name)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "per_dataset_results.json")
    with open(out_path, "w") as f:
        json.dump(all_res, f, indent=2)
    print(f"\n  {model_name} per-dataset results → {out_path}")
    return all_res


def run_dataset_wise_evaluation_all(MODEL_LIST, cfg, samples_per_class=100):
    return {name: run_dataset_wise_evaluation(name, cfg, samples_per_class=samples_per_class)
            for name in MODEL_LIST}


# ─────────────────────────────────────────────────────────────────────────
# Example usage:
#
# MODEL_LIST = list(MODEL_REGISTRY.keys())
# dataset_wise_results = run_dataset_wise_evaluation_all(MODEL_LIST, cfg)
# ─────────────────────────────────────────────────────────────────────────


## Run Everything

This is the "expected final output" loop from your spec, wired up to every
stage above. Uncomment once the dataset placeholder is filled in.


In [ ]:
# ── Auto-persist each model's checkpoint the moment it finishes training ───
# Kaggle sessions cap out (9-12h) before all 7 models finish, so instead of
# waiting until the very end (Stage 6/7) to save anything, we package and
# persist EVERY model's checkpoint + logs immediately after that model's
# training completes (and now, after every curriculum STAGE too -- see
# train_pipeline.py / the training loop cell). This way, if the session
# dies/times out mid-model, everything up to that point is already safe.
#
# Two mechanisms (use whichever fits your workflow, or both):
#   1. zip + FileLink  -> one click download in the Notebook's Output pane,
#      available immediately, no setup required.
#   2. Kaggle Dataset API push -> uploads the zip to a persistent Kaggle
#      Dataset in your account (prajwalm2213/fer-model). Survives even if
#      you close the tab or the session crashes, since it lives on Kaggle's
#      servers, not the kernel. Requires the Kaggle Legacy API key added as
#      Kaggle Secrets (one-time setup, done below).

import os
import shutil
import zipfile
import json
from IPython.display import FileLink, display

# ── One-time auth setup: load your Legacy API key from Kaggle Secrets ─────
# Add-ons -> Secrets -> add KAGGLE_USERNAME and KAGGLE_KEY (from the
# kaggle.json you got via "Create Legacy API Key" on kaggle.com/settings).
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
    print("✅ Kaggle credentials loaded from Secrets.")
except Exception as e:
    print(f"⚠️  Could not load Kaggle Secrets ({e}). "
          f"Dataset auto-push will be skipped until KAGGLE_USERNAME/KAGGLE_KEY are set.")

PERSIST_DATASET_SLUG = "prajwalm2213/fer-model"  # <- your dataset

def package_model_artifacts(model_name, cfg):
    """Zip every checkpoint + log file belonging to model_name into one archive."""
    out_dir = "/kaggle/working/persisted"
    os.makedirs(out_dir, exist_ok=True)
    zip_path = os.path.join(out_dir, f"{model_name}_artifacts.zip")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for folder in [cfg.CHECKPOINT_DIR, cfg.LOG_DIR]:
            if not os.path.isdir(folder):
                continue
            for fname in os.listdir(folder):
                if model_name in fname:
                    full = os.path.join(folder, fname)
                    zf.write(full, arcname=os.path.join(os.path.basename(folder), fname))

    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"  📦 Packaged {model_name} -> {zip_path} ({size_mb:.1f} MB)")
    return zip_path


def offer_download(zip_path):
    """Show a one-click download link in the notebook Output pane."""
    display(FileLink(zip_path))


def push_to_kaggle_dataset(zip_path, dataset_slug, version_note):
    """
    Push zip_path to an existing Kaggle Dataset as a new version, so the
    checkpoint survives independent of this session.
    """
    if dataset_slug is None:
        return
    stage_dir = "/kaggle/working/_dataset_push"
    os.makedirs(stage_dir, exist_ok=True)
    shutil.copy(zip_path, stage_dir)
    metadata = {
        "title": dataset_slug.split("/")[-1],
        "id": dataset_slug,
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(stage_dir, "dataset-metadata.json"), "w") as f:
        json.dump(metadata, f)
    ret = os.system(
        f'kaggle datasets version -p "{stage_dir}" -m "{version_note}" -r zip --dir-mode zip'
    )
    if ret == 0:
        print(f"  ☁️  Pushed to Kaggle Dataset: {dataset_slug}")
    else:
        print(f"  ⚠️  Kaggle Dataset push failed (exit {ret}) — check API credentials.")


def persist_model(model_name, cfg, dataset_slug=PERSIST_DATASET_SLUG):
    """Call this right after a model (or a model's stage) finishes."""
    zip_path = package_model_artifacts(model_name, cfg)
    offer_download(zip_path)
    push_to_kaggle_dataset(zip_path, dataset_slug, version_note=f"{model_name} checkpoint")

print("Persistence utilities loaded ✓")


## Restore progress from a previous session (run this FIRST if resuming)

Kaggle wipes `/kaggle/working` on every new session, so a restarted session has no memory of models you already trained. If you downloaded `train_progress.json` and any `*_artifacts.zip` files, upload them as a Kaggle Dataset and attach it here, then run the cell below **before** the "Run Training" cell.

In [ ]:
# ── OPTIONAL: Restore progress from a previous session (robust version) ────
# Handles: renamed files (Kaggle appends " (1)", " (4)" etc. on duplicate
# uploads), nested subfolders, AND already-unzipped checkpoints/logs folders
# (not just *_artifacts.zip archives).

import glob, shutil, zipfile, json, os

RESTORE_INPUT_DIR = "/kaggle/input/models/prajwalm2213/convnext-vit/pytorch/default/1"   # <-- EDIT ME each session

if os.path.isdir(RESTORE_INPUT_DIR):
    restored_any = False
    os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
    os.makedirs("/kaggle/working/logs", exist_ok=True)

    # Debug: show what's actually in there, so path/name issues are obvious
    print(f"📂 Contents of {RESTORE_INPUT_DIR}:")
    for root, dirs, files in os.walk(RESTORE_INPUT_DIR):
        depth = root.replace(RESTORE_INPUT_DIR, "").count(os.sep)
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/")
        for f in files:
            print(f"{indent}  {f}")
    print()

    # 1) restore train_progress*.json (any renamed variant), recursively
    prog_matches = glob.glob(os.path.join(RESTORE_INPUT_DIR, "**", "train_progress*.json"), recursive=True)
    if prog_matches:
        # if multiple versions got uploaded, merge their "completed" lists
        # together rather than guessing which one is newest
        merged = {"completed": [], "best_acc": {}, "ckpt_path": {}, "stage_done": {}}
        for p in prog_matches:
            with open(p) as f:
                d = json.load(f)
            for m in d.get("completed", []):
                if m not in merged["completed"]:
                    merged["completed"].append(m)
            merged["best_acc"].update(d.get("best_acc", {}))
            merged["ckpt_path"].update(d.get("ckpt_path", {}))
            # stage_done: keep the HIGHEST stage seen per model across merged
            # files, so a model partway through Stage 3 doesn't get treated
            # as only Stage 1 done just because an older progress file is
            # merged in after a newer one.
            for m, s in d.get("stage_done", {}).items():
                merged["stage_done"][m] = max(s, merged["stage_done"].get(m, 0))
        with open("/kaggle/working/train_progress.json", "w") as f:
            json.dump(merged, f, indent=2)
        print(f"✅ Restored train_progress.json (merged from {len(prog_matches)} file(s)) "
              f"-> completed: {merged['completed']}")
        restored_any = True

    # 2) restore any *_artifacts.zip, anywhere under the input dir
    zip_matches = glob.glob(os.path.join(RESTORE_INPUT_DIR, "**", "*_artifacts.zip"), recursive=True)
    for zip_src in zip_matches:
        with zipfile.ZipFile(zip_src, "r") as zf:
            zf.extractall("/kaggle/working")
        print(f"✅ Unzipped {os.path.basename(zip_src)}")
        restored_any = True

    # 3) restore already-unzipped checkpoints/ and logs/ folders, anywhere nested
    for want, dest in [("checkpoints", "/kaggle/working/checkpoints"),
                        ("logs", "/kaggle/working/logs")]:
        for src_dir in glob.glob(os.path.join(RESTORE_INPUT_DIR, "**", want), recursive=True):
            if os.path.isdir(src_dir):
                for fname in os.listdir(src_dir):
                    shutil.copy(os.path.join(src_dir, fname), os.path.join(dest, fname))
                print(f"✅ Copied {len(os.listdir(src_dir))} file(s) from {src_dir} -> {dest}")
                restored_any = True

    if not restored_any:
        print(f"⚠️  Still nothing found under {RESTORE_INPUT_DIR}. Check the printed tree above "
              f"and compare it against this path -- the mounted folder name might not match "
              f"the slug you typed.")
    else:
        with open("/kaggle/working/train_progress.json") as f:
            _p = json.load(f)
        print(f"\n📋 Progress restored -- {len(_p['completed'])} model(s) already done: {_p['completed']}")
        print("You can now run the 'Run Training' cell below; it will skip these and start on the next model.")
else:
    print(f"ℹ️  RESTORE_INPUT_DIR ({RESTORE_INPUT_DIR}) not found. Check the exact mounted "
          f"path under /kaggle/input/ (list /kaggle/input/ to see it) and edit the path above.")

📂 Contents of /kaggle/input/models/prajwalm2213/convnext-vit/pytorch/default/1:
1/
  ConvNeXt_ViT_training_log.json
  train_progress.json
  ConvNeXt_ViT_artifacts/
    checkpoints/
      ConvNeXt_ViT_stage1_best.pth
      ConvNeXt_ViT_stage3_best.pth
      ConvNeXt_ViT_stage2_best.pth
    logs/
      ConvNeXt_ViT_training_log.json

✅ Restored train_progress.json (merged from 1 file(s)) -> completed: ['SACNN_ALSTM', 'SACNN_ViT', 'ConvNeXt_ViT']


In [ ]:
# ── Run Training — ALL MODELS, ONE AFTER ANOTHER (no manual re-run needed) ──
# This cell loops through every un-finished model in MODEL_LIST in a single
# execution, training model after model with no manual re-run in between.
#
# 12-HOUR SESSION CAP — HOW THIS HANDLES IT:
# Kaggle's per-session GPU cap (9-12h) is fixed regardless of activity, and
# GPU notebooks can't be auto-scheduled to restart themselves (Kaggle's
# scheduler only supports CPU notebooks) — so starting the next session is
# still a manual step for you. What this cell minimizes is how much work
# that next session has to redo:
#   - Checkpoints now persist after every STAGE (not just after a model's
#     all 3 stages finish), via on_stage_done -> persist_model(). So if the
#     session dies mid-Stage-3 of model 5, models 1-4 AND model 5's Stage 1
#     + Stage 2 checkpoints are already safe.
#   - stage_done is tracked per model in train_progress.json, so the next
#     session resumes model 5 at Stage 3 -- it does NOT retrain Stages 1-2.
#   - Each persist step still prints a one-click local-download link, and
#     (if PERSIST_DATASET_SLUG is set above) also auto-pushes to a Kaggle
#     Dataset with zero clicks.
#
# What YOU do when a session ends with models still remaining:
#   1. Download the latest zips (or rely on the auto Kaggle Dataset push).
#   2. Start a new Kaggle session, attach those artifacts as a Dataset input
#      (see "Restore progress" cell above), point RESTORE_INPUT_DIR at it.
#   3. Run "Save & Run All (Commit)" again -- this cell will read
#      train_progress.json, see which model/stage it stopped at, and
#      continue exactly from there.

import json

MODEL_LIST = list(MODEL_REGISTRY.keys())   # all 7, or e.g. ["Proposed", "SACNN_ALSTM"]
PROGRESS_FILE = "/kaggle/working/train_progress.json"

def _load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            d = json.load(f)
    else:
        d = {}
    d.setdefault("completed", [])
    d.setdefault("best_acc", {})
    d.setdefault("ckpt_path", {})
    d.setdefault("stage_done", {})   # {model_name: last fully-completed stage int}
    return d

def _save_progress(progress):
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)

progress = _load_progress()

def _on_stage_done(model_name, stage_num, cfg):
    """Fires right after each curriculum stage finishes -- persist immediately
    so a session death mid-model doesn't lose the stages that DID finish."""
    progress["stage_done"][model_name] = stage_num
    _save_progress(progress)
    print(f"  💾 Persisting {model_name} after Stage {stage_num}...")
    persist_model(model_name, cfg)

remaining = [m for m in MODEL_LIST if m not in progress["completed"]]

if not remaining:
    print(f"✅ All {len(MODEL_LIST)} models in MODEL_LIST are already trained:")
    for m in MODEL_LIST:
        print(f"   - {m}: best_acc={progress['best_acc'].get(m, float('nan')):.4f}")
    print("\nNothing left to train. Move on to the 'Finalize' cell to run Stages 3-7.")
else:
    # Dataloaders are cheap to rebuild but avoid re-creating them if this is
    # a continuation within the same live kernel session.
    if 'train_loader' not in globals() or 'val_loader' not in globals() or 'test_loader' not in globals():
        train_loader, val_loader, test_loader = get_dataloaders()
        assert len(train_loader.dataset) > 0, (
            "train_loader is EMPTY. Your dataset paths in the Config cell are wrong "
            "or the folder names inside train/ don't match RAFDBDataset.FOLDER_TO_IDX. "
            "Run the diagnostic snippet (list cfg.RAF_DB_DIR, then its train/ subfolder) "
            "before continuing.")
        assert len(val_loader.dataset) > 0, (
            "val_loader is EMPTY -- same issue, check the RAF-DB 'test' split folder.")
        print(f"Sanity check passed: train={len(train_loader.dataset)} samples, "
              f"val={len(val_loader.dataset)} samples.\n")

    print(f"── Training {len(remaining)} model(s) this run: {remaining} ──\n")

    for _name in remaining:
        _pos = len(progress["completed"]) + 1
        _resume_stage = progress["stage_done"].get(_name, 0) + 1
        print(f"\n{'='*70}")
        print(f"── Model {_pos}/{len(MODEL_LIST)}: {_name}"
              + (f"  (resuming at Stage {_resume_stage})" if _resume_stage > 1 else "")
              + " ──")
        print(f"{'='*70}\n")

        best_acc, ckpt_path, full_log = train_model(
            _name, train_loader, val_loader, cfg,
            resume_stage=_resume_stage, on_stage_done=_on_stage_done,
        )

        progress["completed"].append(_name)
        progress["best_acc"][_name] = float(best_acc)
        progress["ckpt_path"][_name] = str(ckpt_path)
        _save_progress(progress)

        left = [m for m in MODEL_LIST if m not in progress["completed"]]
        print(f"\n✅ {_name} fully done -- best_acc={best_acc:.4f}.")
        if left:
            print(f"👉 Continuing automatically to the next model: {left[0]} "
                  f"({len(left)} remaining: {left})")
        else:
            print("\n🎉 All models trained! Move on to the 'Finalize' cell to run Stages 3-7.")


## Finalize (Stages 3-7) — run once every model above is trained

In [ ]:
# ── Finalize — Stages 3-7 (run only after ALL models are trained) ──────────
# This will refuse to run until every model in MODEL_LIST has a completed
# entry in train_progress.json, so you can't accidentally evaluate/benchmark
# against a model that never finished training.

progress = _load_progress()
_not_done = [m for m in MODEL_LIST if m not in progress["completed"]]
assert not _not_done, (
    f"Not all models are trained yet -- still missing: {_not_done}. "
    f"Go back and re-run the 'Run Training' cell above until every model in "
    f"MODEL_LIST is done, then re-run this cell."
)

if 'train_loader' not in globals() or 'val_loader' not in globals() or 'test_loader' not in globals():
    train_loader, val_loader, test_loader = get_dataloaders()

# train_summary is rebuilt from disk for downstream reference (best_acc / ckpt_path).
# Per-model training curves/logs are read from cfg.LOG_DIR by plot_all_training_curves,
# not from this dict, so it's fine that full per-epoch logs aren't kept across restarts.
train_summary = {
    m: {"best_acc": progress["best_acc"][m], "ckpt_path": progress["ckpt_path"][m]}
    for m in MODEL_LIST
}

# Stage 3 -- evaluate every architecture on the held-out test set
eval_summary = evaluate_all_models(MODEL_LIST, cfg, test_loader)

# Stage 4 -- training/validation curves per model
curve_paths = plot_all_training_curves(MODEL_LIST, cfg)

# Stage 5 -- benchmark every architecture (params, FLOPs, speed)
bench_summary = benchmark_all_models(MODEL_LIST, cfg)

# Stage 6 -- comparison tables (CSV + Excel)
perf_df, eff_df = generate_comparison_tables(eval_summary, bench_summary, cfg)

# Stage 7 -- ablation study (component ablations on the Proposed model)
ablation_configs = {
    "Without SA-CNN":  ("ALSTM_ViT",   train_loader, val_loader),
    "Without ALSTM":   ("SACNN_ViT",   train_loader, val_loader),
    "Without ViT":     ("SACNN_ALSTM", train_loader, val_loader),
}
ablation_results = run_ablation_study(ablation_configs, cfg, test_loader)
ablation_df = ablation_summary(ablation_results, eval_summary["Proposed"])

# Dataset-wise breakdown (JAFFE / CK+ / RAF-DB / AffectNet / FER-2013)
dataset_wise_results = run_dataset_wise_evaluation_all(MODEL_LIST, cfg)

print("\n✅ All stages complete.")
print("perf_df, eff_df, ablation_df, dataset_wise_results are ready for the")
print("Discussion / Summary section below, and for building the figures shown")
print("in your Results and Discussion outline (bar charts, confusion matrices,")
print("ROC curves, accuracy/loss curves) from perf_df / eval_summary directly.")


## Comparative Performance Chart (all 7 models, matches the TOC's "Comparative Performance Analysis")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics = ["Accuracy (%)", "Precision (macro %)", "Recall (macro %)", "F1 (macro %)"]
x = np.arange(len(perf_df))
width = 0.2
for i, m in enumerate(metrics):
    axes[0].bar(x + i*width, perf_df[m], width, label=m)
axes[0].set_xticks(x + width*1.5)
axes[0].set_xticklabels(perf_df["Model"], rotation=45, ha="right")
axes[0].set_title("Accuracy / Precision / Recall / F1 by Model")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", alpha=0.3)

merged = perf_df.merge(eff_df, on="Model")
axes[1].scatter(merged["Total Params"] / 1e6, merged["Accuracy (%)"], s=90, color="#2E86AB")
for _, row in merged.iterrows():
    axes[1].annotate(row["Model"], (row["Total Params"]/1e6, row["Accuracy (%)"]),
                     fontsize=8, xytext=(5, 5), textcoords="offset points")
axes[1].set_xlabel("Parameters (M)")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Accuracy vs. Model Size")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{cfg.EVAL_DIR}/comparative_performance.png", dpi=150, bbox_inches="tight")
plt.show()
perf_df


## Experimental Setup

In [ ]:
from IPython.display import display, Markdown, Image

setup_md = f"""
| Item | Value |
|---|---|
| Compound emotion classes | {cfg.NUM_CLASSES} — {", ".join(cfg.COMPOUND_CLASSES)} |
| Basic emotion classes (source) | {", ".join(cfg.BASIC_CLASSES)} |
| Image size | {cfg.IMAGE_SIZE} x {cfg.IMAGE_SIZE} |
| Batch size | {cfg.BATCH_SIZE} |
| Optimizer | AdamW (lr={cfg.LEARNING_RATE}, weight_decay={cfg.WEIGHT_DECAY}) |
| LR schedule | OneCycleLR, 3-stage curriculum (frozen -> partial unfreeze -> full unfreeze) |
| Early stopping patience | {cfg.PATIENCE} epochs |
| Embedding dim / LSTM hidden / layers | {cfg.EMBED_DIM} / {cfg.LSTM_HIDDEN} / {cfg.LSTM_LAYERS} |
| ViT backbone | {cfg.VIT_MODEL_NAME} |
| Seed | {cfg.SEED} |
| Device | {cfg.DEVICE} |
| Models compared | {len(MODEL_LIST)} — {", ".join(MODEL_LIST)} |
| Training set size (combined, incl. synthetic splicing) | {len(train_loader.dataset)} |
| Validation / Test set size | {len(val_loader.dataset)} |
"""
display(Markdown(setup_md))


## Performance Evaluation of Each Architecture

One static section per model -- matches "Performance Evaluation of SA-CNN + ALSTM",
"... SA-CNN + ViT", "... ConvNeXt + ViT", "... EfficientNetV2 + ViT",
"... ConvNeXt + Mamba", "... ALSTM + ViT", and "... the Proposed Model" from
your outline. Each shows the metrics table plus that model's confusion
matrix, ROC curves, and PR curves (already saved by Stage 3).


In [ ]:
MODEL_DISPLAY_NAMES = {
    "SACNN_ALSTM":   "SA-CNN + ALSTM",
    "SACNN_ViT":     "SA-CNN + Vision Transformer (ViT)",
    "ConvNeXt_ViT":  "ConvNeXt + Vision Transformer (ViT)",
    "EffNetV2_ViT":  "EfficientNetV2 + Vision Transformer (ViT)",
    "ConvNeXt_Mamba":"ConvNeXt + Mamba",
    "ALSTM_ViT":     "ALSTM + Vision Transformer (ViT)",
    "Proposed":      "Proposed Model (Morphological Splicing -> SA-CNN -> ALSTM -> Vision Transformer)",
}


### Performance Evaluation of SA-CNN + ALSTM


In [ ]:
name = "SACNN_ALSTM"
m = eval_summary[name]

metrics_md = f"""
| Metric | Value |
|---|---|
| Accuracy | {m['accuracy']:.2f}% |
| Precision (macro / weighted) | {m['precision_macro']:.2f}% / {m['precision_weighted']:.2f}% |
| Recall (macro / weighted) | {m['recall_macro']:.2f}% / {m['recall_weighted']:.2f}% |
| F1-Score (macro / weighted) | {m['f1_macro']:.2f}% / {m['f1_weighted']:.2f}% |
| Sensitivity (macro) | {m['sensitivity_macro']:.2f}% |
| Specificity (macro) | {m['specificity_macro']:.2f}% |
| AUC (macro) | {m['auc_macro']:.2f}% |
"""
display(Markdown(metrics_md))

for label, key in [("Confusion Matrix", "confusion_matrix"),
                    ("ROC Curves", "roc_curve"),
                    ("Precision-Recall Curves", "pr_curve")]:
    path = m["paths"][key]
    if os.path.exists(path):
        print(f"  {label}:")
        display(Image(filename=path))


### Performance Evaluation of SA-CNN + Vision Transformer (ViT)


In [ ]:
name = "SACNN_ViT"
m = eval_summary[name]

metrics_md = f"""
| Metric | Value |
|---|---|
| Accuracy | {m['accuracy']:.2f}% |
| Precision (macro / weighted) | {m['precision_macro']:.2f}% / {m['precision_weighted']:.2f}% |
| Recall (macro / weighted) | {m['recall_macro']:.2f}% / {m['recall_weighted']:.2f}% |
| F1-Score (macro / weighted) | {m['f1_macro']:.2f}% / {m['f1_weighted']:.2f}% |
| Sensitivity (macro) | {m['sensitivity_macro']:.2f}% |
| Specificity (macro) | {m['specificity_macro']:.2f}% |
| AUC (macro) | {m['auc_macro']:.2f}% |
"""
display(Markdown(metrics_md))

for label, key in [("Confusion Matrix", "confusion_matrix"),
                    ("ROC Curves", "roc_curve"),
                    ("Precision-Recall Curves", "pr_curve")]:
    path = m["paths"][key]
    if os.path.exists(path):
        print(f"  {label}:")
        display(Image(filename=path))


### Performance Evaluation of ConvNeXt + Vision Transformer (ViT)


In [ ]:
name = "ConvNeXt_ViT"
m = eval_summary[name]

metrics_md = f"""
| Metric | Value |
|---|---|
| Accuracy | {m['accuracy']:.2f}% |
| Precision (macro / weighted) | {m['precision_macro']:.2f}% / {m['precision_weighted']:.2f}% |
| Recall (macro / weighted) | {m['recall_macro']:.2f}% / {m['recall_weighted']:.2f}% |
| F1-Score (macro / weighted) | {m['f1_macro']:.2f}% / {m['f1_weighted']:.2f}% |
| Sensitivity (macro) | {m['sensitivity_macro']:.2f}% |
| Specificity (macro) | {m['specificity_macro']:.2f}% |
| AUC (macro) | {m['auc_macro']:.2f}% |
"""
display(Markdown(metrics_md))

for label, key in [("Confusion Matrix", "confusion_matrix"),
                    ("ROC Curves", "roc_curve"),
                    ("Precision-Recall Curves", "pr_curve")]:
    path = m["paths"][key]
    if os.path.exists(path):
        print(f"  {label}:")
        display(Image(filename=path))


### Performance Evaluation of EfficientNetV2 + Vision Transformer (ViT)


In [ ]:
name = "EffNetV2_ViT"
m = eval_summary[name]

metrics_md = f"""
| Metric | Value |
|---|---|
| Accuracy | {m['accuracy']:.2f}% |
| Precision (macro / weighted) | {m['precision_macro']:.2f}% / {m['precision_weighted']:.2f}% |
| Recall (macro / weighted) | {m['recall_macro']:.2f}% / {m['recall_weighted']:.2f}% |
| F1-Score (macro / weighted) | {m['f1_macro']:.2f}% / {m['f1_weighted']:.2f}% |
| Sensitivity (macro) | {m['sensitivity_macro']:.2f}% |
| Specificity (macro) | {m['specificity_macro']:.2f}% |
| AUC (macro) | {m['auc_macro']:.2f}% |
"""
display(Markdown(metrics_md))

for label, key in [("Confusion Matrix", "confusion_matrix"),
                    ("ROC Curves", "roc_curve"),
                    ("Precision-Recall Curves", "pr_curve")]:
    path = m["paths"][key]
    if os.path.exists(path):
        print(f"  {label}:")
        display(Image(filename=path))


### Performance Evaluation of ConvNeXt + Mamba


In [ ]:
name = "ConvNeXt_Mamba"
m = eval_summary[name]

metrics_md = f"""
| Metric | Value |
|---|---|
| Accuracy | {m['accuracy']:.2f}% |
| Precision (macro / weighted) | {m['precision_macro']:.2f}% / {m['precision_weighted']:.2f}% |
| Recall (macro / weighted) | {m['recall_macro']:.2f}% / {m['recall_weighted']:.2f}% |
| F1-Score (macro / weighted) | {m['f1_macro']:.2f}% / {m['f1_weighted']:.2f}% |
| Sensitivity (macro) | {m['sensitivity_macro']:.2f}% |
| Specificity (macro) | {m['specificity_macro']:.2f}% |
| AUC (macro) | {m['auc_macro']:.2f}% |
"""
display(Markdown(metrics_md))

for label, key in [("Confusion Matrix", "confusion_matrix"),
                    ("ROC Curves", "roc_curve"),
                    ("Precision-Recall Curves", "pr_curve")]:
    path = m["paths"][key]
    if os.path.exists(path):
        print(f"  {label}:")
        display(Image(filename=path))


### Performance Evaluation of ALSTM + Vision Transformer (ViT)


In [ ]:
name = "ALSTM_ViT"
m = eval_summary[name]

metrics_md = f"""
| Metric | Value |
|---|---|
| Accuracy | {m['accuracy']:.2f}% |
| Precision (macro / weighted) | {m['precision_macro']:.2f}% / {m['precision_weighted']:.2f}% |
| Recall (macro / weighted) | {m['recall_macro']:.2f}% / {m['recall_weighted']:.2f}% |
| F1-Score (macro / weighted) | {m['f1_macro']:.2f}% / {m['f1_weighted']:.2f}% |
| Sensitivity (macro) | {m['sensitivity_macro']:.2f}% |
| Specificity (macro) | {m['specificity_macro']:.2f}% |
| AUC (macro) | {m['auc_macro']:.2f}% |
"""
display(Markdown(metrics_md))

for label, key in [("Confusion Matrix", "confusion_matrix"),
                    ("ROC Curves", "roc_curve"),
                    ("Precision-Recall Curves", "pr_curve")]:
    path = m["paths"][key]
    if os.path.exists(path):
        print(f"  {label}:")
        display(Image(filename=path))


### Performance Evaluation of Proposed Model (Morphological Splicing -> SA-CNN -> ALSTM -> Vision Transformer)


In [ ]:
name = "Proposed"
m = eval_summary[name]

metrics_md = f"""
| Metric | Value |
|---|---|
| Accuracy | {m['accuracy']:.2f}% |
| Precision (macro / weighted) | {m['precision_macro']:.2f}% / {m['precision_weighted']:.2f}% |
| Recall (macro / weighted) | {m['recall_macro']:.2f}% / {m['recall_weighted']:.2f}% |
| F1-Score (macro / weighted) | {m['f1_macro']:.2f}% / {m['f1_weighted']:.2f}% |
| Sensitivity (macro) | {m['sensitivity_macro']:.2f}% |
| Specificity (macro) | {m['specificity_macro']:.2f}% |
| AUC (macro) | {m['auc_macro']:.2f}% |
"""
display(Markdown(metrics_md))

for label, key in [("Confusion Matrix", "confusion_matrix"),
                    ("ROC Curves", "roc_curve"),
                    ("Precision-Recall Curves", "pr_curve")]:
    path = m["paths"][key]
    if os.path.exists(path):
        print(f"  {label}:")
        display(Image(filename=path))


## Dataset-wise Performance Analysis (JAFFE, CK+, RAF-DB, AffectNet, FER-2013)

In [ ]:
rows = []
for model_name, ds_results in dataset_wise_results.items():
    for r in ds_results:
        rows.append({
            "Model": model_name, "Dataset": r["dataset"],
            "N": r["n_samples"], "Accuracy (%)": r["accuracy"],
            "Weighted F1 (%)": r["weighted_f1"], "Macro F1 (%)": r["macro_f1"],
        })
dataset_wise_df = pd.DataFrame(rows)

if len(dataset_wise_df):
    pivot = dataset_wise_df.pivot(index="Model", columns="Dataset", values="Accuracy (%)")
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Dataset-wise Accuracy by Model")
    ax.legend(title="Dataset", fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{cfg.EVAL_DIR}/dataset_wise_accuracy.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("  ⚠  dataset_wise_df is empty — check dataset pool paths in Config.")

dataset_wise_df


## Accuracy, Precision, Recall, and F1-Score Analysis

In [ ]:
try:
    display(perf_df.style.background_gradient(cmap="Blues", subset=[
        "Accuracy (%)", "Precision (macro %)", "Recall (macro %)", "F1 (macro %)"
    ]))
except Exception:
    display(perf_df)

fig, ax = plt.subplots(figsize=(12, 6))
metrics = ["Accuracy (%)", "Precision (macro %)", "Recall (macro %)", "F1 (macro %)"]
x = np.arange(len(perf_df)); width = 0.2
for i, m in enumerate(metrics):
    ax.bar(x + i*width, perf_df[m], width, label=m)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(perf_df["Model"], rotation=45, ha="right")
ax.set_ylabel("%")
ax.set_title("Accuracy / Precision / Recall / F1-Score — All Models")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{cfg.EVAL_DIR}/accuracy_precision_recall_f1.png", dpi=150, bbox_inches="tight")
plt.show()


## Confusion Matrix Analysis

In [ ]:
n = len(MODEL_LIST)
cols = 3
rows_n = math.ceil(n / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(6*cols, 5*rows_n))
axes = np.array(axes).reshape(-1)

most_confused_overall = {}
for i, name in enumerate(MODEL_LIST):
    cm = np.array(eval_summary[name]["confusion_matrix"])
    off_diag = cm.copy()
    np.fill_diagonal(off_diag, 0)
    r, c = np.unravel_index(off_diag.argmax(), off_diag.shape)
    pair = (cfg.COMPOUND_CLASSES[r], cfg.COMPOUND_CLASSES[c])
    most_confused_overall[name] = (pair, int(off_diag[r, c]))

    im = axes[i].imshow(cm, cmap="Blues")
    axes[i].set_title(name, fontsize=10, fontweight="bold")
    axes[i].set_xlabel("Predicted"); axes[i].set_ylabel("True")
for j in range(n, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.savefig(f"{cfg.EVAL_DIR}/confusion_matrix_grid.png", dpi=150, bbox_inches="tight")
plt.show()

print("Most-confused compound-emotion pair per model:")
for name, (pair, count) in most_confused_overall.items():
    print(f"  {name:16s}: {pair[0]!r} <-> {pair[1]!r}  ({count} samples)")


## ROC Curve Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
aucs = [eval_summary[name]["auc_macro"] for name in MODEL_LIST]
bars = ax.bar(MODEL_LIST, aucs, color="#2E86AB")
ax.set_ylabel("Macro-average AUC (%)")
ax.set_title("Macro AUC by Model")
plt.xticks(rotation=45, ha="right")
for b, v in zip(bars, aucs):
    ax.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.1f}", ha="center", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{cfg.EVAL_DIR}/macro_auc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Per-model ROC curve plots (per-class + macro-average) were shown above,")
print("under each model's own 'Performance Evaluation' section.")


## Training and Validation Performance (Accuracy and Loss Curves)

In [ ]:
for name in MODEL_LIST:
    display(Markdown(f"### {MODEL_DISPLAY_NAMES.get(name, name)}"))
    paths = curve_paths.get(name, {})
    for key, label in [("accuracy_curve", "Accuracy"), ("loss_curve", "Loss")]:
        p = paths.get(key)
        if p and os.path.exists(p):
            display(Image(filename=p))
        else:
            print(f"  ⚠  {label} curve not available for {name}")


## Ablation Study — Results

In [ ]:
display(ablation_df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2E86AB" if d >= 0 else "#E84855" for d in ablation_df["Δ Accuracy"]]
ax.bar(ablation_df["Configuration"], ablation_df["Accuracy (%)"], color=colors)
ax.axhline(ablation_df.loc[ablation_df["Configuration"]=="Proposed (Full Model)", "Accuracy (%)"].values[0],
           color="gray", linestyle="--", linewidth=1, label="Proposed (Full Model)")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Ablation Study — Component Contribution")
plt.xticks(rotation=20, ha="right")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{cfg.EVAL_DIR}/ablation_study.png", dpi=150, bbox_inches="tight")
plt.show()


## Computational Performance Analysis (Training Time, Inference Time, Model Parameters, and FLOPs)

In [ ]:
display(eff_df)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].bar(eff_df["Model"], eff_df["Total Params"] / 1e6, color="#2E86AB")
axes[0,0].set_title("Total Parameters (M)"); axes[0,0].set_ylabel("Params (M)")

axes[0,1].bar(eff_df["Model"], eff_df["FLOPs (G)"], color="#5B8C5A")
axes[0,1].set_title("FLOPs (G)"); axes[0,1].set_ylabel("GFLOPs")

axes[1,0].bar(eff_df["Model"], eff_df["Training Time (s)"], color="#E8A24C")
axes[1,0].set_title("Training Time (s)"); axes[1,0].set_ylabel("Seconds")

axes[1,1].bar(eff_df["Model"], eff_df["Inference Time (ms/img)"], color="#E84855")
axes[1,1].set_title("Inference Time (ms/img)"); axes[1,1].set_ylabel("ms")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{cfg.EVAL_DIR}/computational_performance.png", dpi=150, bbox_inches="tight")
plt.show()


## Discussion of Results

In [ ]:
best_acc_row       = perf_df.iloc[0]
worst_acc_row      = perf_df.iloc[-1]
most_efficient_row = eff_df.sort_values("Total Params").iloc[0]
best_auc_name       = max(MODEL_LIST, key=lambda n: eval_summary[n]["auc_macro"])
best_auc_value       = eval_summary[best_auc_name]["auc_macro"]
auc_matches_accuracy = (best_auc_name == best_acc_row["Model"])

has_proposed = "Proposed" in perf_df["Model"].values
if has_proposed:
    proposed_row  = perf_df[perf_df["Model"] == "Proposed"].iloc[0]
    proposed_rank = int(perf_df.reset_index(drop=True).index[perf_df["Model"] == "Proposed"][0]) + 1
else:
    proposed_row, proposed_rank = None, None

precision_recall_gap = abs(best_acc_row["Precision (macro %)"] - best_acc_row["Recall (macro %)"])
balance_note = "a well-balanced classifier" if precision_recall_gap < 3 else "a precision/recall trade-off worth noting"
auc_note = ("consistent with the accuracy ranking above" if auc_matches_accuracy else
            "a different ranking than raw accuracy -- suggesting it separates classes by "
            "confidence better than it converts that separation into top-1 predictions")

# Ablation deltas (skip the first row, which is the Proposed baseline with delta 0)
if len(ablation_df) > 1:
    deltas          = ablation_df.iloc[1:]
    worst_component = deltas.loc[deltas["Δ Accuracy"].idxmin(), "Configuration"]
    worst_delta     = deltas["Δ Accuracy"].min()
    best_component  = deltas.loc[deltas["Δ Accuracy"].idxmax(), "Configuration"]
    best_delta      = deltas["Δ Accuracy"].max()
    ablation_note = (
        f"**{worst_component}** caused the largest accuracy drop from the full Proposed "
        f"model ({worst_delta:.2f} pts), indicating that component contributes the most "
        f"to the Proposed model's performance. **{best_component}** caused the smallest "
        f"drop ({best_delta:.2f} pts), suggesting that component is comparatively less "
        f"critical -- though still not redundant, since removing it still underperforms "
        f"the full model."
    )
else:
    ablation_note = "Ablation results were not available at generation time."

proposed_note = (
    f"The Proposed model (Morphological Splicing -> SA-CNN -> ALSTM -> ViT) ranked "
    f"**#{proposed_rank} of {len(MODEL_LIST)}** by accuracy, at {proposed_row['Accuracy (%)']:.2f}%."
    if has_proposed else ""
)

discussion = f"""
**Comparative Performance Analysis.** Across the {len(MODEL_LIST)} architectures
evaluated, **{best_acc_row['Model']}** achieved the highest accuracy at
**{best_acc_row['Accuracy (%)']:.2f}%** (F1 macro {best_acc_row['F1 (macro %)']:.2f}%),
while **{worst_acc_row['Model']}** trailed at **{worst_acc_row['Accuracy (%)']:.2f}%**.

{proposed_note}

**Accuracy / Precision / Recall / F1.** See the grouped bar chart above for the
full per-model breakdown; {best_acc_row['Model']} leads on accuracy, and its
precision/recall balance (P={best_acc_row['Precision (macro %)']:.2f}%,
R={best_acc_row['Recall (macro %)']:.2f}%) indicates {balance_note}.

**Confusion Matrix Analysis.** The most-confused compound-emotion pair per
model is listed above; recurring confusions across multiple architectures
point to compound classes that share overlapping basic-emotion components
rather than a weakness specific to any one model.

**ROC Curve Analysis.** **{best_auc_name}** achieved the highest macro AUC
({best_auc_value:.2f}%), {auc_note}.

**Computational Performance.** **{most_efficient_row['Model']}** is the most
parameter-efficient model at {most_efficient_row['Total Params']/1e6:.1f}M
parameters. Weighing accuracy against cost: {best_acc_row['Model']} is the
top performer, but {most_efficient_row['Model']} is the better choice if
deployment constraints (latency, memory) matter more than the last few
points of accuracy.

**Ablation Study.** {ablation_note}
"""
display(Markdown(discussion))


## Summary of Results

In [ ]:
bottom_line = f"{best_acc_row['Model']} gives the strongest raw accuracy in this comparison."
if has_proposed and best_acc_row["Model"] != "Proposed":
    bottom_line += (
        f" The Proposed architecture ranks #{proposed_rank} overall -- trading a small "
        f"amount of accuracy for the interpretability/efficiency benefits of its "
        f"splicing-based training strategy."
    )
bottom_line += f" Use {most_efficient_row['Model']} instead if inference cost is the binding constraint."

proposed_rank_line = (
    f"#{proposed_rank} of {len(MODEL_LIST)} by accuracy ({proposed_row['Accuracy (%)']:.2f}%)"
    if has_proposed else "N/A"
)

summary = f"""
- **Best overall accuracy:** {best_acc_row['Model']} — {best_acc_row['Accuracy (%)']:.2f}% accuracy, {best_acc_row['F1 (macro %)']:.2f}% F1 (macro).
- **Best AUC:** {best_auc_name} — {best_auc_value:.2f}% macro AUC.
- **Most efficient:** {most_efficient_row['Model']} — {most_efficient_row['Total Params']/1e6:.1f}M params, {most_efficient_row['Inference Time (ms/img)']:.2f} ms/img inference.
- **Proposed model rank:** {proposed_rank_line}
- **Ablation contributions:** removing any single branch (SA-CNN, ALSTM, or ViT) from the Proposed model reduces accuracy versus the full 3-branch fusion — see the Ablation Study table above for exact deltas per component.
- **Dataset generalization:** see the Dataset-wise Performance Analysis table/chart above for how each model transfers across JAFFE, CK+, RAF-DB, AffectNet, and FER-2013.

**Bottom line:** {bottom_line}
"""
display(Markdown(summary))

print("\n" + "="*70)
print("  ALL RESULTS AND DISCUSSION SECTIONS GENERATED")
print("="*70)
print("  Every chart/table above was also saved as a PNG/CSV/XLSX under:")
print(f"  {cfg.EVAL_DIR}/")
